# Architettura 3: Bottleneck Denso / Fully-Connected
## Studio e Sperimentazione AE-XAD su Dataset BUSI

### 📌 Cosa fa la Baseline (Riferimento)
La Baseline mantiene un flusso interamente convoluzionale con volume latente $64 \times 28 \times 28$ (50.176 valori) e circa 1,6 milioni di parametri addestrabili.

### ⚙️ Cosa fa questa Architettura a differenza della Baseline
* **Bottleneck Denso (FC):** Sostituisce la continuità convoluzionale appiattendo l'output dell'encoder ($50.176$ valori), comprimendolo a una rappresentazione latente unidimensionale di soli **256 neuroni** tramite un layer lineare *fully-connected*, e riespandendolo a $50.176$ con un secondo layer lineare prima del reshape a $64 \times 28 \times 28$.
* **Esplosione Parametrica:** I parametri addestrabili salgono da 1,6 milioni a **~27,3 milioni** (circa 17 volte la baseline).

### ⚠️ Motivazione dell'Esperimento e Causa del Fallimento (Overfitting)
* **Obiettivo:** Testare l'efficacia di un bottleneck concettualmente analogo a quello degli autoencoder classici fortemente compressi.
* **Analisi del Fallimento:** A fronte delle sole 93 immagini normali di training disponibili nel dataset BUSI, un modello con oltre 27 milioni di parametri soffre di un **grave overfitting strutturale**.
* **Risultati:** L'AUC crolla a **$0{,}8137 \pm 0{,}0256$** a 200 epoche (la peggiore del benchmark) e degrada drasticamente nel tempo fino a $0{,}808$ a 250 epoche, confermando empiricamente che gli strati densi ad alta dimensionalità non sono adatti a regimi di dati limitati in ambito clinico.


In [ ]:
import sys
from pathlib import Path

# Project root resolution (works both from repo root and experiments/ folder)
PROJECT_ROOT = Path("..").resolve() if (Path("..") / "repo").exists() else Path(".").resolve()
REPO_DIR = PROJECT_ROOT / "repo" / "AE-XAD-main"

if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

print(f"Project root: {PROJECT_ROOT}")
print(f"AE-XAD repo: {REPO_DIR}")


In [ ]:
# Clone repo AE-XAD e applicazione patch ===

import sys
import subprocess
from pathlib import Path

REPO_DIR = PROJECT_ROOT
PATCH_MARKER = REPO_DIR / ".patched"  # file sentinella: esiste solo dopo patching completo

# --- Idempotenza ---
# Controlliamo il marker, non solo l'esistenza della cartella:
# un clone interrotto lascerebbe la repo a metà senza marker.
if PATCH_MARKER.exists():
    print(f"✓ Repo già clonata e patchata in {REPO_DIR}. Skip.")
else:
    # Se la cartella esiste ma senza marker, la rimuoviamo e ricloniamo pulita.
    # Questo gestisce il caso di clone interrotto.
    if REPO_DIR.exists():
        print("Cartella repo trovata ma priva di marker patch. Rimozione e reclone...")
        subprocess.run(["rm", "-rf", str(REPO_DIR)], check=True)

    # --- Clone ---
    print("Clonazione repo AE-XAD...")
    result = subprocess.run(
        ["git", "clone",
         "https://github.com/AIDALab-DIMES/AE-XAD.git",
         str(REPO_DIR)],
        capture_output=True, text=True
    )
    if result.returncode != 0:
        raise RuntimeError(f"git clone fallito:\n{result.stderr}")
    print("✓ Clone completato.")

        # PATCH 1: tools/utils.py — import warn → import warnings
        # Il modulo 'warn' non esiste nella stdlib Python.
    # L'autore probabilmente voleva 'warnings' (il modulo standard
    # per emettere avvisi a runtime). Senza questa patch, qualsiasi
    # import di tools.utils crasha con ModuleNotFoundError al primo
    # import, bloccando tutto il pipeline di evaluation.
    utils_path = REPO_DIR / "tools" / "utils.py"
    utils_src = utils_path.read_text()

    if "import warn\n" not in utils_src:
        raise RuntimeError(
            "Patch 1: riga 'import warn' non trovata in tools/utils.py. "
            "La repo potrebbe essere cambiata upstream — verifica manualmente."
        )

    utils_patched = utils_src.replace("import warn\n", "import warnings\n", 1)
    utils_patched = utils_patched.replace("warn.warn(", "warnings.warn(", 1)
    utils_path.write_text(utils_patched)
    print("✓ Patch 1 applicata: tools/utils.py (import warn → import warnings)")

        # PATCH 2: dataloaders/custom_VGG.py — augmentor IAA → nativi
        # Gli augmentor A.IAA* sono wrapper attorno alla libreria imgaug.
    # imgaug usa np.sctypes che è stato rimosso in NumPy 2.0:
    # su Colab (che usa NumPy 2.x) qualsiasi import di custom_VGG.py
    # crasha a import-time, prima ancora di istanziare il dataloader.
    # Anche se non useremo CustomVGGAD_CP direttamente (useremo la
    # nostra CustomGrayscaleAD), il modulo viene importato dal Trainer
    # e il crash si propagherebbe lo stesso.
    #
    # Sostituzioni:
    #   A.IAAAdditiveGaussianNoise → A.GaussNoise
    #   A.IAAPiecewiseAffine       → A.ElasticTransform(p=0.3)
    #   A.IAASharpen               → A.Sharpen
    #   A.IAAEmboss                → A.Emboss
    #
    # Semantica: non identica all'originale, ma equivalente per il
    # nostro scopo. ElasticTransform è più potente di PiecewiseAffine
    # ma produce deformazioni locali comparabili. GaussNoise, Sharpen,
    # Emboss sono drop-in replacement diretti.
    custom_vgg_path = REPO_DIR / "dataloaders" / "custom_VGG.py"
    vgg_src = custom_vgg_path.read_text()

    replacements = [
        ("A.IAAAdditiveGaussianNoise", "A.GaussNoise"),
        ("A.IAAPiecewiseAffine",       "A.ElasticTransform(p=0.3)"),
        ("A.IAASharpen",               "A.Sharpen"),
        ("A.IAAEmboss",                "A.Emboss"),
    ]

    for old, new in replacements:
        if old not in vgg_src:
            print(f"  WARN: '{old}' non trovato — già patchato o repo cambiata upstream.")
        else:
            vgg_src = vgg_src.replace(old, new)
            print(f"  ✓ {old} → {new}")

    custom_vgg_path.write_text(vgg_src)
    print("✓ Patch 2 applicata: dataloaders/custom_VGG.py (augmentor IAA → nativi)")

    # --- Marker: scriviamo il sentinella solo se tutto è andato bene ---
    # Se una patch fallisce con eccezione, il marker non viene scritto
    # e la prossima esecuzione rieseguirà tutto da capo. Questo è il
    # comportamento corretto: un patching parziale è peggio di nessun patching.
    PATCH_MARKER.write_text("patched\n")
    print("✓ Marker patch scritto.")

# --- Aggiungi repo al sys.path ---
# Necessario perché i moduli della repo (loss, trainer, ecc.) usano
# import relativi alla root del progetto. Senza questo, Python non li trova.
# Usiamo insert(0, ...) invece di append per dare priorità alla repo
# rispetto a eventuali pacchetti installati con lo stesso nome.
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
    print(f"✓ {REPO_DIR} aggiunto a sys.path")
else:
    print(f"✓ {REPO_DIR} già in sys.path")

# --- Verifica import ---
# Test veloce: se questi import non crashano, le patch hanno funzionato.
try:
    import tools.utils  # avrebbe crashato con 'import warn' non patchato
    print("✓ tools.utils importabile")
except Exception as e:
    raise RuntimeError(f"Import tools.utils fallito dopo patch: {e}")

In [ ]:
# Costanti di preprocessing e funzioni helper ===

import os
import numpy as np
from pathlib import Path
from PIL import Image
from typing import Optional

# -----------------------------------------------------------------------
# COSTANTI GLOBALI
# -----------------------------------------------------------------------
# Centralizzare le costanti qui serve a due scopi:
# 1. Modificare un valore (es. la risoluzione) si fa in un posto solo
# 2. Le celle successive le leggono da qui, rendendo esplicite le dipendenze

RESIZE_TO           = (224,224)   # test con 224

SPLIT_SEED          = 12345        # seed fisso per la divisione train/test dei normali
                                   # garantisce riproducibilità: stessi file in train
                                   # ad ogni esecuzione

NORMAL_TRAIN_RATIO  = 0.70         # 70% dei normali va in train, 30% in test
                                   # scelta del briefing: bilancia quantità di dati
                                   # normali per il training con campioni di test sufficienti

DRIVE_BASE = PROJECT_ROOT
BUSI_RAW   = DRIVE_BASE / "busi_raw"
OUT_DIR    = DRIVE_BASE / "busi_prepared"

# -----------------------------------------------------------------------
# FUNZIONI HELPER
# -----------------------------------------------------------------------

def load_image_grayscale(path: Path) -> Image.Image:
    """
    Carica un'immagine come grayscale a 1 canale (mode='L').

    Perché mode='L' e non 'RGB'?
    Le ecografie BUSI sono già grayscale nel contenuto, ma alcuni file PNG
    sono salvati come RGB o RGBA (3-4 canali tutti uguali). Forzare 'L'
    garantisce che l'array risultante abbia shape (H, W) e non (H, W, 3),
    indipendentemente da come il file è stato salvato originariamente.
    Replicare su 3 canali (convert('RGB')) sarebbe sbagliato per noi:
    il modello è adattato a 1 canale, triplicare il dato non aggiunge
    informazione e spreca memoria.
    """
    return Image.open(path).convert("L")


def load_mask(image_path: Path) -> Optional[np.ndarray]:

    """
    Carica la maschera binaria associata a un'immagine anomala.

    Cerca il file seguendo il pattern: {basename}_mask.png
    Esempio: 'benign (1).png' → 'benign (1)_mask.png'

    Restituisce:
      - array NumPy HxW con valori {0, 1} se la maschera esiste e non è vuota
      - None se il file non esiste o la maschera è tutta a zero

    Perché restituire None invece di un array di zeri?
    La loss AEXAD_loss_norm_vgg calcola lambda_p = N_pixel / sum(gt).
    Se sum(gt) == 0 → divisione per zero → loss = inf → training diverge.
    Un'immagine senza maschera valida non può stare nel training set.
    Meglio escluderla esplicitamente (con log) che includerla silenziosamente.
    """
    
    stem = image_path.stem
    folder = image_path.parent
    # maschera principale + eventuali _mask_1, _mask_2, ...
    mask_files = [folder / f"{stem}_mask.png"]
    mask_files += sorted(folder.glob(f"{stem}_mask_*.png"))
    mask_files = [m for m in mask_files if m.exists()]
    if not mask_files:
        return None
    merged = None
    for mp in mask_files:
        m = (np.array(Image.open(mp).convert("L")) > 0).astype(np.uint8)
        merged = m if merged is None else (merged | m)   # OR logico
    if merged.sum() == 0:
        return None
    return merged


def save_image_grayscale(pil_img: Image.Image, dest_path: Path) -> None:
    """
    Ridimensiona a RESIZE_TO con interpolazione BILINEAR e salva come PNG.

    Perché BILINEAR per le immagini?
    BILINEAR interpola i valori dei pixel con una media pesata dei 4 vicini.
    Per immagini a toni di grigio produce bordi morbidi e naturali, senza
    aliasing. È la scelta standard per il downscaling di immagini fotografiche.
    NEAREST sarebbe sbagliato qui: produrrebbe un effetto a blocchi visibile.
    """
    dest_path.parent.mkdir(parents=True, exist_ok=True)
    pil_img.resize(RESIZE_TO, Image.BILINEAR).save(dest_path)


def save_mask(mask_arr: np.ndarray, dest_path: Path) -> None:
    """
    Ridimensiona la maschera binaria a RESIZE_TO con NEAREST e salva come PNG.

    Perché NEAREST per le maschere?
    La maschera è binaria {0, 1}. Qualsiasi interpolazione 'intelligente'
    (BILINEAR, BICUBIC) introdurrebbe valori intermedi (es. 0.3, 0.7) che
    non hanno senso semantico e farebbero perdere la binarietà.
    NEAREST preserva i valori esatti: un pixel è o normale o anomalo, non
    'un po' anomalo'.

    Prima di salvare, rimoltiplichiamo per 255 per avere {0, 255} nel file PNG:
    è il formato atteso dal dataloader (valori {0, 255}, non {0, 1}).
    """
    dest_path.parent.mkdir(parents=True, exist_ok=True)
    mask_255 = (mask_arr * 255).astype(np.uint8)
    Image.fromarray(mask_255.astype(np.uint8)).resize(RESIZE_TO, Image.NEAREST).save(dest_path)


def find_raw_folders(raw_dir: Path) -> dict[str, Path]:
    """
    Cerca le cartelle normal/, benign/, malignant/ dentro raw_dir.

    Usa os.walk invece di glob diretto perché la struttura interna dei
    dataset Kaggle dopo unzip non è garantita: a volte c'è una cartella
    intermedia con il nome del dataset, a volte no.
    os.walk esplora ricorsivamente e trova le cartelle target indipendentemente
    dal livello di nesting.

    Restituisce un dict {'normal': Path, 'benign': Path, 'malignant': Path}.
    Solleva FileNotFoundError se una delle tre cartelle non viene trovata.
    """
    targets = {"normal", "benign", "malignant"}
    found: dict[str, Path] = {}

    for root, dirs, _ in os.walk(raw_dir):
        for d in dirs:
            if d.lower() in targets:
                found[d.lower()] = Path(root) / d

    missing = targets - found.keys()
    if missing:
        raise FileNotFoundError(
            f"Cartelle non trovate in {raw_dir}: {missing}\n"
            f"Trovate: {list(found.keys())}"
        )

    return found


def collect_images(folder: Path) -> list[Path]:
    """
    Elenca tutti i file PNG in folder, escludendo le maschere.

    Le maschere hanno '_mask' nel nome del file: le escludiamo per ottenere
    solo le immagini originali. Senza questo filtro, le maschere verrebbero
    trattate come immagini e duplicate nel dataset.

    Restituisce lista ordinata (sort per riproducibilità cross-platform:
    os.listdir non garantisce ordine deterministico su tutti i filesystem).
    """
    return sorted([
        p for p in folder.glob("*.png")
        if "_mask" not in p.name
    ])


# --- Verifica rapida delle costanti ---
print(f"✓ RESIZE_TO:          {RESIZE_TO}")
print(f"✓ SPLIT_SEED:         {SPLIT_SEED}")
print(f"✓ NORMAL_TRAIN_RATIO: {NORMAL_TRAIN_RATIO}")
print(f"✓ BUSI_RAW:           {BUSI_RAW}")
print(f"✓ OUT_DIR:            {OUT_DIR}")
print("\n✓ Funzioni helper definite: load_image_grayscale, load_mask,")
print("  save_image_grayscale, save_mask, find_raw_folders, collect_images")

In [ ]:
# Discovery cartelle sorgente e statistiche per classe ===

from pathlib import Path

# --- Ricerca delle cartelle nel dataset grezzo ---
# find_raw_folders esplora ricorsivamente busi_raw/ e restituisce
# {'normal': Path, 'benign': Path, 'malignant': Path}.
# Se una cartella manca, crasha con FileNotFoundError esplicito.
raw_folders = find_raw_folders(BUSI_RAW)

print("Cartelle trovate:")
for cls, path in raw_folders.items():
    print(f"  {cls:12s} → {path}")

# --- Raccolta liste immagini per classe ---
# collect_images filtra via le maschere (_mask nel nome) e restituisce
# solo le immagini originali, ordinate per nome.
images_by_class: dict[str, list[Path]] = {
    cls: collect_images(path)
    for cls, path in raw_folders.items()
}

# --- Statistiche ---
print("\nStatistiche dataset grezzo:")
print(f"  {'Classe':<12} {'Immagini':>10}")
print(f"  {'-'*24}")
total = 0
for cls, imgs in images_by_class.items():
    print(f"  {cls:<12} {len(imgs):>10}")
    total += len(imgs)
print(f"  {'-'*24}")
print(f"  {'TOTALE':<12} {total:>10}")

# --- Verifica maschere per le classi anomale ---
# Per benign e malignant, contiamo quante immagini hanno una maschera
# valida (non None). Questo ci dà già un'idea di quante anomalie
# sopravviveranno al filtro della cella 10.
print("\nVerifica maschere (classi anomale):")
for cls in ("benign", "malignant"):
    folder = raw_folders[cls]
    imgs   = images_by_class[cls]
    n_with_mask    = sum(1 for p in imgs if (folder / (p.stem + "_mask.png")).exists())
    n_nonempty     = sum(
        1 for p in imgs
        if load_mask(p) is not None
    )
    print(f"  {cls:<12} — con file maschera: {n_with_mask:>3} / {len(imgs)}"
          f"  |  maschera non vuota: {n_nonempty:>3} / {len(imgs)}")

print("\n✓ Discovery completata.")

In [ ]:
# Split dei normali (train/test) ===

import numpy as np

# --- Lista immagini normali ---
# Partiamo dalla lista già raccolta in cella 7.
# Le immagini normali non hanno maschere: sono il riferimento di "normalità"
# su cui l'autoencoder impara a ricostruire bene.
normal_images: list[Path] = images_by_class["normal"]
n_normal = len(normal_images)

# --- Split deterministico con seed fisso ---
# Usiamo np.random.default_rng invece del vecchio np.random.seed + shuffle:
# default_rng è l'API moderna di NumPy, isolata dallo stato globale del RNG.
# Questo significa che eseguire altre celle con operazioni random prima o dopo
# non altera questo split. Con il vecchio API basterebbe una chiamata a
# np.random.randint altrove per spostare tutto lo split.
rng = np.random.default_rng(SPLIT_SEED)

# Creiamo un array di indici e lo mescoliamo in-place.
# Lavoriamo su indici invece che sulla lista direttamente per avere
# controllo esplicito su quale porzione va in train e quale in test.
indices = np.arange(n_normal)
rng.shuffle(indices)

# --- Calcolo soglia di taglio ---
# int() tronca: con 133 immagini e ratio 0.70 → 93.1 → 93 train, 40 test.
# Arrotondare per eccesso (round) darebbe 93 ugualmente, ma troncare è più
# conservativo: preferibile avere qualche immagine in più nel test set.
n_train = int(n_normal * NORMAL_TRAIN_RATIO)
n_test  = n_normal - n_train

train_indices = indices[:n_train]
test_indices  = indices[n_train:]

# Liste esplicite di Path: più chiare di array di indici nelle celle successive
train_normal: list[Path] = [normal_images[i] for i in train_indices]
test_normal:  list[Path] = [normal_images[i] for i in test_indices]

# --- Verifica ---
assert len(train_normal) + len(test_normal) == n_normal, "Split incompleto: immagini perse"
assert len(set(train_normal) & set(test_normal)) == 0,   "Overlap tra train e test"

print(f"Normali totali:  {n_normal}")
print(f"  → train:       {len(train_normal)}  ({len(train_normal)/n_normal*100:.1f}%)")
print(f"  → test:        {len(test_normal)}   ({len(test_normal)/n_normal*100:.1f}%)")
print(f"\n✓ Split deterministico con SPLIT_SEED={SPLIT_SEED}")

In [ ]:
# Salvataggio immagini normali (train + test) ===

import time

# --- Struttura cartelle di destinazione ---
# Le normali non hanno maschere: salviamo solo le immagini.
# La struttura rispecchia quella attesa da extract_dataset della repo:
# una cartella per split (train/test), una sottocartella per classe.
TRAIN_NORMAL_DIR = OUT_DIR / "train" / "normal"
TEST_NORMAL_DIR  = OUT_DIR / "test"  / "normal"

TRAIN_NORMAL_DIR.mkdir(parents=True, exist_ok=True)
TEST_NORMAL_DIR.mkdir(parents=True, exist_ok=True)

def process_normal_split(
    images: list[Path],
    dest_dir: Path,
    split_name: str
) -> None:
    """
    Carica, ridimensiona e salva le immagini normali in dest_dir.

    Ogni immagine viene salvata con lo stesso nome del file originale.
    Preservare il nome originale è importante per la tracciabilità:
    se in fase di debug vogliamo risalire a un'immagine specifica nel
    dataset grezzo, il nome è il collegamento diretto.
    """
    t0 = time.time()
    for i, src_path in enumerate(images):
        dest_path = dest_dir / src_path.name
        # Idempotenza: se il file esiste già (cella rilancaita),
        # saltiamo il resize/salvataggio. Il resize è deterministico
        # quindi il risultato sarebbe identico, ma risparmiamo I/O su Drive.
        if dest_path.exists():
            continue
        pil = load_image_grayscale(src_path)
        save_image_grayscale(pil, dest_path)

    elapsed = time.time() - t0
    n_saved = len(list(dest_dir.glob("*.png")))
    print(f"  {split_name:<6} → {dest_dir}")
    print(f"           {n_saved} file presenti  ({elapsed:.1f}s)")

print("Salvataggio immagini normali...")
process_normal_split(train_normal, TRAIN_NORMAL_DIR, "train")
process_normal_split(test_normal,  TEST_NORMAL_DIR,  "test")

# --- Verifica finale ---
n_train_saved = len(list(TRAIN_NORMAL_DIR.glob("*.png")))
n_test_saved  = len(list(TEST_NORMAL_DIR.glob("*.png")))

assert n_train_saved == len(train_normal), (
    f"Train: attesi {len(train_normal)} file, trovati {n_train_saved}"
)
assert n_test_saved == len(test_normal), (
    f"Test: attesi {len(test_normal)} file, trovati {n_test_saved}"
)

print(f"\n✓ Normali salvati: {n_train_saved} train + {n_test_saved} test = "
      f"{n_train_saved + n_test_saved} totali")

In [ ]:
# Processing anomalie (benign e malignant) ===

import time
from dataclasses import dataclass

# --- Struttura cartelle di destinazione ---
# Le anomalie hanno sia immagini che maschere: creiamo sottocartelle
# separate per images e masks dentro ogni classe.
# Questa separazione è richiesta dalla funzione extract_dataset della repo
# e rende esplicito a colpo d'occhio cosa contiene ogni cartella.
for split in ("train", "test"):
    for cls in ("benign", "malignant"):
        (OUT_DIR / split / cls / "images").mkdir(parents=True, exist_ok=True)
        (OUT_DIR / split / cls / "masks").mkdir(parents=True, exist_ok=True)

# Nel briefing tutte le anomalie (tranne quelle selezionate per il training,
# cella 18) vanno nel test set. La selezione dei 3+3 per il training avviene
# in cella 18 al momento del lancio, non qui.
# Qui salviamo TUTTE le anomalie valide in test/, pronte per essere usate.
ANOM_DEST: dict[str, dict[str, Path]] = {
    cls: {
        "images": OUT_DIR / "test" / cls / "images",
        "masks":  OUT_DIR / "test" / cls / "masks",
    }
    for cls in ("benign", "malignant")
}

# -----------------------------------------------------------------------

@dataclass
class ProcessResult:
    cls:     str
    saved:   list[Path]
    skipped: list[tuple[Path, str]]  # (path, motivo)


def process_anomaly_class(
    cls: str,
    images: list[Path],
) -> ProcessResult:
    """
    Processa tutte le immagini anomale di una classe.

    Per ogni immagine:
      1. Cerca la maschera con load_mask (restituisce None se assente o vuota)
      2. Se None → skip con log del motivo
      3. Altrimenti salva immagine e maschera nelle cartelle di destinazione

    Perché skippare invece di usare una maschera zero?
    La loss AEXAD_loss_norm_vgg calcola:
        lambda_p = N_pixel / sum(gt)
    Se sum(gt) == 0 → lambda_p = inf → loss = nan → training diverge
    immediatamente. Non c'è modo di recuperare un'immagine senza maschera
    valida nel training set di questo metodo.
    """
    img_dest  = ANOM_DEST[cls]["images"]
    mask_dest = ANOM_DEST[cls]["masks"]

    saved:   list[Path]               = []
    skipped: list[tuple[Path, str]]   = []

    t0 = time.time()
    for src_path in images:

        # Idempotenza: se entrambi i file esistono già, saltiamo
        dest_img  = img_dest  / src_path.name
        dest_mask = mask_dest / src_path.name
        if dest_img.exists() and dest_mask.exists():
            saved.append(dest_img)
            continue

        # Carica maschera: None se mancante o vuota
        mask = load_mask(src_path)

        if mask is None:
            # Distinguiamo i due casi per un log più informativo
            mask_path = src_path.parent / (src_path.stem + "_mask.png")
            motivo = "file maschera assente" if not mask_path.exists() else "maschera vuota (sum=0)"
            skipped.append((src_path, motivo))
            continue

        # Salva immagine e maschera ridimensionate
        pil = load_image_grayscale(src_path)
        save_image_grayscale(pil, dest_img)
        save_mask(mask, dest_mask)
        saved.append(dest_img)

    elapsed = time.time() - t0
    print(f"\n  [{cls}]  {elapsed:.1f}s")
    print(f"    salvate:  {len(saved)}")
    print(f"    skippate: {len(skipped)}")
    if skipped:
        for path, motivo in skipped:
            print(f"    [SKIP] {path.name} — {motivo}")

    return ProcessResult(cls=cls, saved=saved, skipped=skipped)


# --- Esecuzione ---
print("Processing anomalie...")
result_benign    = process_anomaly_class("benign",    images_by_class["benign"])
result_malignant = process_anomaly_class("malignant", images_by_class["malignant"])

# --- Riepilogo finale ---
total_saved   = len(result_benign.saved)   + len(result_malignant.saved)
total_skipped = len(result_benign.skipped) + len(result_malignant.skipped)

print(f"\n{'='*40}")
print(f"Totale salvate:  {total_saved}")
print(f"Totale skippate: {total_skipped}")

# Verifica di coerenza con i dati della cella 7:
# dalla cella 7 sappiamo che tutte le maschere sono valide,
# quindi non ci aspettiamo skip. Se ce ne sono, è un problema.
assert total_skipped == 0, (
    f"{total_skipped} immagini skippate inaspettatamente. "
    f"Controlla il log sopra per i dettagli."
)

print("✓ Nessuno skip — coerente con la verifica maschere di cella 7.")

In [ ]:
# Sanity check ===

import random
import numpy as np
from PIL import Image
from pathlib import Path

random.seed(SPLIT_SEED)  # riproducibilità nella scelta del campione random

# -----------------------------------------------------------------------
# Verifichiamo tre proprietà fondamentali su un campione per ogni classe:
#   1. La coppia (immagine, maschera) esiste su disco
#   2. Immagine e maschera hanno la stessa shape
#   3. I valori della maschera sono in {0, 255} e nient'altro
#   4. L'immagine è grayscale a 1 canale (shape (H,W), non (H,W,3))
#
# Perché fare questo check esplicitamente?
# Gli errori di preprocessing silenziosi sono i più pericolosi:
# un'immagine salvata come RGB invece di grayscale non causerebbe
# crash immediato, ma produrrebbe tensori di shape sbagliata nel
# dataloader, con un errore criptico durante il training.
# Meglio scoprirlo adesso con un assert chiaro.
# -----------------------------------------------------------------------

def check_pair(img_path: Path, mask_path: Path, label: str) -> None:
    """Verifica una coppia (immagine, maschera) e solleva AssertionError se qualcosa non va."""

    # 1. Esistenza
    assert img_path.exists(),  f"[{label}] Immagine non trovata: {img_path}"
    assert mask_path.exists(), f"[{label}] Maschera non trovata: {mask_path}"

    img_arr  = np.array(Image.open(img_path))
    mask_arr = np.array(Image.open(mask_path))

    # 2. Shape uguale
    assert img_arr.shape == mask_arr.shape, (
        f"[{label}] Shape mismatch: immagine {img_arr.shape} vs maschera {mask_arr.shape}"
    )

    # 3. Valori maschera in {0, 255}
    unique_vals = set(np.unique(mask_arr).tolist())
    assert unique_vals.issubset({0, 255}), (
        f"[{label}] Valori maschera inattesi: {unique_vals} (attesi sottoinsieme di {{0, 255}})"
    )

    # 4. Immagine grayscale: shape deve essere (H, W), non (H, W, 3)
    assert img_arr.ndim == 2, (
        f"[{label}] Immagine non grayscale: shape {img_arr.shape} "
        f"(atteso 2D, trovato {img_arr.ndim}D)"
    )

    # 5. Risoluzione corretta dopo resize
    assert img_arr.shape == RESIZE_TO, (
        f"[{label}] Risoluzione errata: {img_arr.shape} (atteso {RESIZE_TO})"
    )

    print(f"  ✓ [{label}]  shape={img_arr.shape}  "
          f"mask_vals={sorted(unique_vals)}  "
          f"img_range=[{img_arr.min()}, {img_arr.max()}]")


# --- Check normali (train e test) ---
# I normali non hanno maschere: verifichiamo solo shape e grayscale
print("Normali:")
for split, folder, lst in [
    ("train", OUT_DIR / "train" / "normal", train_normal),
    ("test",  OUT_DIR / "test"  / "normal", test_normal),
]:
    sample = random.choice(lst)
    img_path = folder / sample.name
    assert img_path.exists(), f"[normal/{split}] File non trovato: {img_path}"
    arr = np.array(Image.open(img_path))
    assert arr.ndim == 2, f"[normal/{split}] Immagine non grayscale: shape {arr.shape}"
    assert arr.shape == RESIZE_TO, f"[normal/{split}] Risoluzione errata: {arr.shape}"
    print(f"  ✓ [normal/{split}]  shape={arr.shape}  "
          f"range=[{arr.min()}, {arr.max()}]")

# --- Check anomalie (benign e malignant, solo test) ---
print("\nAnomalie:")
for cls in ("benign", "malignant"):
    img_dir  = OUT_DIR / "test" / cls / "images"
    mask_dir = OUT_DIR / "test" / cls / "masks"
    all_imgs = sorted(img_dir.glob("*.png"))
    sample   = random.choice(all_imgs)
    mask_path = mask_dir / sample.name
    check_pair(sample, mask_path, f"{cls}/test")

print("\n✓ Sanity check superato su tutti i campioni.")

In [ ]:
# --- Verifica range valori immagini BUSI ---
from PIL import Image
import numpy as np
from pathlib import Path

sample_dirs = [
    PROJECT_ROOT,
    PROJECT_ROOT,
]

for d in sample_dirs:
    files = sorted(d.glob("*.png"))[:3]
    for f in files:
        arr = np.array(Image.open(f))
        print(f"{f.name}: dtype={arr.dtype}, min={arr.min()}, max={arr.max()}, shape={arr.shape}")

In [ ]:
# Salvataggio split_info.json e statistiche finali ===

import json
import time
from pathlib import Path

# -----------------------------------------------------------------------
# Perché salvare split_info.json?
# Il notebook può essere rilanciato in sessioni diverse. Senza un file
# che registra quali immagini sono finite in train e quali in test,
# non potremmo ricostruire lo split esatto (anche con seed fisso,
# dovremmo rieseguire tutto dall'inizio). split_info.json è la memoria
# permanente dello split: basta leggerlo per sapere esattamente cosa
# c'è in ogni set, senza rieseguire le celle 7-10.
# -----------------------------------------------------------------------

split_info = {
    "metadata": {
        "split_seed":         SPLIT_SEED,
        "normal_train_ratio": NORMAL_TRAIN_RATIO,
        "resize_to":          list(RESIZE_TO),
        "created_at":         time.strftime("%Y-%m-%dT%H:%M:%S"),
    },
    "train": {
        "normal": [p.name for p in train_normal],
    },
    "test": {
        "normal":    [p.name for p in test_normal],
        "benign":    sorted([p.name for p in (OUT_DIR / "test" / "benign" / "images").glob("*.png")]),
        "malignant": sorted([p.name for p in (OUT_DIR / "test" / "malignant" / "images").glob("*.png")]),
    },
    "skipped": {
        "benign":    [(p.name, m) for p, m in result_benign.skipped],
        "malignant": [(p.name, m) for p, m in result_malignant.skipped],
    },
    "counts": {
        "train_normal":    len(train_normal),
        "test_normal":     len(test_normal),
        "test_benign":     len(result_benign.saved),
        "test_malignant":  len(result_malignant.saved),
        "skipped_benign":  len(result_benign.skipped),
        "skipped_malignant": len(result_malignant.skipped),
    }
}

out_json = OUT_DIR / "split_info.json"
out_json.write_text(json.dumps(split_info, indent=2, ensure_ascii=False))
print(f"✓ split_info.json salvato in {out_json}")

# --- Statistiche finali ---
print("\n" + "="*50)
print("STATISTICHE FINALI — busi_prepared/")
print("="*50)

counts = split_info["counts"]
print(f"\n  TRAIN")
print(f"    normal:         {counts['train_normal']:>4}")
print(f"\n  TEST")
print(f"    normal:         {counts['test_normal']:>4}")
print(f"    benign:         {counts['test_benign']:>4}")
print(f"    malignant:      {counts['test_malignant']:>4}")
print(f"    totale anomalie:{counts['test_benign'] + counts['test_malignant']:>4}")
print(f"\n  SKIPPED")
print(f"    benign:         {counts['skipped_benign']:>4}")
print(f"    malignant:      {counts['skipped_malignant']:>4}")

# Verifica struttura cartelle su disco
print("\n" + "="*50)
print("STRUTTURA CARTELLE — busi_prepared/")
print("="*50)
for p in sorted(OUT_DIR.rglob("*")):
    if p.is_dir():
        n_png = len(list(p.glob("*.png")))
        if n_png > 0:
            print(f"  {str(p.relative_to(OUT_DIR)):<35} {n_png:>4} PNG")

# Verifica che split_info.json sia leggibile e integro
reloaded = json.loads(out_json.read_text())
assert reloaded["counts"] == split_info["counts"], "split_info.json corrotto dopo scrittura"
print(f"\n✓ split_info.json verificato (round-trip JSON ok)")

In [ ]:
# Architettura — variante Arch 3 (dense bottleneck) ===
# ResNet → encoder conv baseline → [flatten → dense → reshape] → decoder baseline.
# UNICA differenza dalla baseline: il bottleneck convoluzionale (passthrough 64×28×28)
# è sostituito da un bottleneck DENSO (Flatten → Linear → Linear → reshape).
# I 3 layer encoder e i 3 stadi decoder restano IDENTICI alla baseline (vincolo rispettato).
#
# La classe MANTIENE il nome ResNet_CNN_Attn_Gray: launch_gray la istanzia per nome
# (ResNet_CNN_Attn_Gray(dim=[1])), quindi NON serve toccare CELLA 15 né CELLA 19.
# bottleneck_dim ha un default → la firma (dim=[1]) usata da launch_gray funziona intatta.
#
# Controprova (relatore scettico): 17× i parametri della baseline su 93 img di training,
# spatial locality distrutta dal flatten. Atteso: pixel-AUC più bassa, heatmap sfocate.

import torch
import torch.nn as nn
from torchvision.models import resnet50, ResNet50_Weights
from einops import rearrange  # noqa: F401 — coerenza con la repo

class ResNet_CNN_Attn_Gray(nn.Module):
    def __init__(self, dim: list, bottleneck_dim: int = 256):
        super().__init__()
        self.dim = dim
        self.bottleneck_dim = bottleneck_dim   # registrato per tracciabilità

        # --- Frozen backbone (identico alla baseline) ---
        encoder_pre = resnet50(weights=ResNet50_Weights.DEFAULT)
        for param in encoder_pre.parameters():
            param.requires_grad = False
        gray_weights = encoder_pre.conv1.weight.data.sum(dim=1, keepdim=True)
        self.conv1 = nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3, bias=False)
        self.conv1.weight.data = gray_weights
        for param in self.conv1.parameters():
            param.requires_grad = False
        self.bn1      = encoder_pre.bn1
        self.relu1    = encoder_pre.relu
        self.maxpool1 = encoder_pre.maxpool
        self.layer1   = encoder_pre.layer1
        self.layer2   = encoder_pre.layer2

        # --- Encoder conv trainable (INTOCCATO: 512→256→128→64, stride 1) ---
        self.encoder = nn.Sequential(
            nn.Conv2d(512, 256, (3, 3), stride=1, padding=1), nn.BatchNorm2d(256), nn.ReLU(),
            nn.Conv2d(256, 128, (3, 3), stride=1, padding=1), nn.BatchNorm2d(128), nn.ReLU(),
            nn.Conv2d(128,  64, (3, 3), stride=1, padding=1), nn.BatchNorm2d(64),
        )

        # --- Bottleneck DENSO — unica differenza dalla baseline ---
        self._flat_dim = 64 * 28 * 28   # 50176
        self.dense_enc = nn.Sequential(
            nn.Flatten(),                              # (B,64,28,28) → (B,50176)
            nn.Linear(self._flat_dim, bottleneck_dim), # → (B, bottleneck_dim)
            nn.ReLU(),
        )
        self.dense_dec = nn.Sequential(
            nn.Linear(bottleneck_dim, self._flat_dim), # → (B,50176)
            nn.ReLU(),
        )

        # --- Decoder baseline (INTOCCATO) ---
        self.up1  = nn.Upsample(scale_factor=2)
        self.up2  = nn.Upsample(scale_factor=2)
        self.up3  = nn.Upsample(scale_factor=2)
        self.tan3 = nn.Tanh()
        self.dec1 = nn.Sequential(
            nn.ConvTranspose2d(64, 32, (4, 4), stride=2, padding=1), nn.SELU(),
            nn.Conv2d(32, 32, (3, 3), stride=1, padding=1), nn.SELU(),
        )
        self.dec2 = nn.Sequential(
            nn.ConvTranspose2d(32, 16, (4, 4), stride=2, padding=1), nn.SELU(),
            nn.Conv2d(16, 16, (3, 3), stride=1, padding=1), nn.SELU(),
        )
        self.dec3 = nn.Sequential(
            nn.ConvTranspose2d(16, 8, (4, 4), stride=2, padding=1), nn.SELU(),
            nn.Conv2d(8, 8, (3, 3), stride=1, padding=1), nn.SELU(),
        )
        self.decoder_final = nn.Sequential(
            nn.Conv2d(8, 8, (3, 3), stride=1, padding=1), nn.SELU(),
            nn.Conv2d(8, dim[0], (3, 3), stride=1, padding=1), nn.ReLU(),
        )

    def stop_finetuning(self):
        for m in (self.conv1, self.bn1, self.maxpool1, self.layer1, self.layer2):
            for p in m.parameters():
                p.requires_grad = False

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        fe  = self.conv1(x)
        fe  = self.bn1(fe)
        fe  = self.relu1(fe)
        fe  = self.maxpool1(fe)
        fe1 = self.layer1(fe)
        fe2 = self.layer2(fe1)

        encoded = self.encoder(fe2)              # (B, 64, 28, 28)

        # --- Bottleneck denso ---
        z     = self.dense_enc(encoded)          # (B, bottleneck_dim)
        z_dec = self.dense_dec(z)                # (B, 50176)
        z_map = z_dec.view(-1, 64, 28, 28)       # (B, 64, 28, 28)  ← -1 NON negoziabile (batch test=1)

        # --- Branch 1: bilineare ---
        up1 = self.up1(z_map)                    # 28→56
        up2 = self.up2(up1)                      # 56→112
        up3 = self.up3(up2)                      # 112→224

        # --- Branch 2: transposed conv ---
        d1 = self.dec1(z_map)                    # 28→56
        d2 = self.dec2(d1)                       # 56→112
        d3 = self.dec3(d2)                       # 112→224

        # --- Attention fusion (identica alla baseline) ---
        d3 = d3 + d3 * torch.sum(self.tan3(up3) ** 2, dim=1).unsqueeze(1)

        return self.decoder_final(d3)


# ----------------------- Verifica -----------------------
print("Verifica ResNet_CNN_Attn_Gray (Arch 3 — dense bottleneck)...")
model = ResNet_CNN_Attn_Gray(dim=[1]).eval()

shapes = {}
def _mk(name):
    def _h(m, i, o): shapes[name] = tuple(o.shape)
    return _h
h1 = model.encoder.register_forward_hook(_mk("encoder"))
h2 = model.dense_enc.register_forward_hook(_mk("z"))

# Test a batch 2 E batch 1 (il test_loader gira a batch_size=1)
for B in (2, 1):
    x = torch.randn(B, 1, 224, 224)
    with torch.no_grad():
        out = model(x)
    assert out.shape == (B, 1, 224, 224), f"output {out.shape} a batch {B}"
h1.remove(); h2.remove()

assert model.conv1.weight.shape == (64, 1, 7, 7)
assert not model.conv1.weight.requires_grad, "conv1 non congelato"
assert shapes["encoder"] == (1, 64, 28, 28), f"encoder {shapes['encoder']}"
assert shapes["z"]       == (1, 256),         f"bottleneck {shapes['z']} (atteso (1,256))"
assert all(p.requires_grad for p in model.dense_enc.parameters())
assert all(p.requires_grad for p in model.dense_dec.parameters())

n_total = sum(p.numel() for p in model.parameters())
n_tr    = sum(p.numel() for p in model.parameters() if p.requires_grad)
n_dense = sum(p.numel() for p in model.dense_enc.parameters()) \
        + sum(p.numel() for p in model.dense_dec.parameters())
print(f"  ✓ Output shape:        {out.shape}")
print(f"  ✓ Bottleneck denso:    {shapes['z']}  (dim={model.bottleneck_dim})")
print(f"  ✓ Parametri totali:    {n_total:,}")
print(f"  ✓ Parametri trainable: {n_tr:,}  (~17× baseline 1,605,505)")
print(f"  ✓ Solo blocco denso:   {n_dense:,}")
print(f"  ✓ Per immagine (93):   {n_tr/93:,.0f} param/img  (baseline ~17,263)")
print("  ⚠ DENSE = controprova: attesa pixel-AUC più bassa, heatmap sfocate.")

del model
print("\n✓ Arch 3 (dense bottleneck) definita e verificata.")

In [ ]:
# Dataloader CustomGrayscaleAD ===

import os
import cv2
import numpy as np
import albumentations as A
from pathlib import Path
from PIL import Image
from torch.utils.data import Dataset
from torchvision import transforms


class CustomGrayscaleAD(Dataset):
    """
    Dataloader per anomaly detection su immagini grayscale a 1 canale.

    Replica l'interfaccia di CustomVGGAD_CP adattandola a:
      - Input a 1 canale invece di 3
      - Normalizzazione scalare (mean=0.0, std=1.0) invece di ImageNet
      - img_size=496 invece di 224
      - gt_label a 1 canale (no repeat(3,1,1))

    Perché NON modificare CustomVGGAD_CP in-place?
    Manteniamo il codice originale funzionante come reference immutabile.
    Se in futuro confrontiamo i risultati grayscale con un run RGB,
    abbiamo le due versioni ben separate e non dobbiamo ricostruire
    lo stato originale della repo.
    """

    def __init__(self, path: str, train: bool = True, img_size: int = 224):
        super().__init__()
        self.img_size = img_size
        self.train = train
        split = "train" if self.train else "test"

        # --- Caricamento array NumPy ---
        # X è salvato come (N, 1, H, W) channel-first (CHW).
        # swapaxes(1,2).swapaxes(2,3) converte in (N, H, W, 1) (HWC).
        # [..., 0] rimuove la dimensione canale: (N, H, W).
        # Risultato: array di immagini grayscale come matrici 2D.
        # Questo è il formato che PIL.Image.fromarray() accetta nativamente
        # per mode='L' senza passaggi aggiuntivi.
        x = (
            np.load(os.path.join(path, f"X_{split}.npy"))
            .swapaxes(1, 2)
            .swapaxes(2, 3)
            .astype(np.uint8)
            [..., 0]                   # (N, H, W, 1) → (N, H, W)
        )
        y  = np.load(os.path.join(path, f"Y_{split}.npy"))
        gt = np.load(os.path.join(path, f"GT_{split}.npy"))

        # GT è (N, 1, H, W): prendiamo gt[:,0] → (N, H, W), valori {0, 255}.
        # Stesso pattern dell'originale: self.gt = gt[:, 0].astype(np.uint8)
        self.gt     = gt[:, 0].astype(np.uint8)
        self.labels = y
        self.images = x  # shape (N, H, W)

        # --- Statistiche del dataset ---
        self.n_norm = int((1 - self.labels).sum())
        self.n_anom = int(self.labels.sum())

        # --- Oversampling delle anomalie ---
        # Il training set ha poche anomalie (6 su ~99 campioni, ~6%).
        # Senza oversampling, il gradiente della loss sarebbe dominato
        # dai normali e il termine anomalo (che guida la localizzazione)
        # avrebbe peso trascurabile. L'oversampling forza il modello
        # a vedere ogni anomalia (repeat_time + reuse_times + 1) volte
        # per epoca rispetto a ogni normale.
        #
        # repeat_time=10: ogni anomalia viene ripetuta 10 volte in più
        # reuse_times=5:  le prime 5 ripetizioni usano l'immagine originale,
        #                 le successive applicano copy_paste (augmentation)
        self.reuse_times = 5
        self.repeat_time = 10

        # anomaly_idx: indici delle anomalie reali in self.images
        # I valori np.arange(...) aggiunti dall'originale sono dead code:
        # il modulo % n_anom in __getitem__ li rende irraggiungibili.
        # Li manteniamo per parità con l'originale.
        self.anomaly_idx = np.append(
            np.argwhere(self.labels == 1).reshape(-1),
            np.arange(
                self.n_norm + self.n_anom,
                self.n_norm + self.n_anom * (self.reuse_times + self.repeat_time),
                1,
            ),
            axis=0,
        )
        self.normal_idx = np.argwhere(self.labels == 0).reshape(-1)

        # --- Normalizzazione ---
        # mean=0.0, std=0.1 scalari: mappa [0,1] → [-1,1].
        # Alternativa: calcolare mean/std specifici di BUSI sul training set
        # (più precisi). Per una baseline è sufficiente il mapping lineare
        # centrato, che non introduce bias verso valori chiari o scuri.
        # Nota: usiamo liste a un elemento [0.5] invece di scalari perché
        # transforms.Normalize richiede sequenze, non scalari.
        self.mean = np.array([0.0])
        self.std  = np.array([1.0])

        # dim: (C, H, W) — usato dal Trainer per inferire la forma dell'input.
        # C=1 per grayscale. H e W sono le dimensioni delle immagini caricate.
        self.dim = (1, self.images.shape[1], self.images.shape[2])

        # --- Pipeline di transform ---
        # ToTensor su PIL mode='L' produce (1, H, W) float [0,1].
        # Normalize([0.5],[0.5]) porta a [-1,1].  SBAGLIATO! [0,1]
        # Non usiamo ColorJitter: opera su canali RGB e non ha senso per grayscale.
        _base = [
            transforms.Resize((self.img_size, self.img_size)),
            transforms.ToTensor(),
            transforms.Normalize([0.0], [1.0]),  # identità: pixel restano in [0,1]  #transforms.Normalize(self.mean.tolist(), self.std.tolist()),
        ]
        self.transform       = transforms.Compose(_base)
        self.transform_mask  = transforms.Compose([transforms.ToTensor()])

        INIZIO AUGMENTATION ===
        self.augmentors = [
            A.HorizontalFlip(p=0.5),
            
            A.OneOf([
                A.GaussNoise(),
                A.GaussNoise(var_limit=(10, 50)),
                A.MultiplicativeNoise(multiplier=(0.9, 1.1), per_channel=False),
            ], p=0.2),
            
            A.OneOf([
                A.MotionBlur(p=0.2),
                A.MedianBlur(blur_limit=3, p=0.1),
                A.Blur(blur_limit=3, p=0.1),
            ], p=0.2),
            
            A.ShiftScaleRotate(
                shift_limit=0.0625, 
                scale_limit=0.2, 
                rotate_limit=10,
                p=0.2
            ),
            
            A.OneOf([
                A.ElasticTransform(p=0.3),
                A.GridDistortion(p=0.1),
            ], p=0.2),
            
            A.OneOf([
                A.CLAHE(clip_limit=2),
                A.Sharpen(),
                A.RandomBrightnessContrast(),
            ], p=0.3),
        ]

        FINE AUGMENTATION ===
        
    # -----------------------------------------------------------------------

    def real_dim(self) -> int:
        return self.n_norm + self.n_anom

    def __len__(self) -> int:
        if self.train:
            # Normali + anomalie originali + oversampling
            return self.n_norm + self.repeat_time * self.n_anom + self.reuse_times * self.n_anom
        else:
            return self.n_norm + self.n_anom

    def __getitem__(self, index: int) -> dict:
        if self.train and index >= self.n_norm + self.n_anom:
            # Indice nel range di oversampling: mappa ciclicamente sulle anomalie
            offset = index - self.n_norm - self.n_anom
            _index = self.anomaly_idx[offset % self.n_anom]
            image       = self.images[_index]   # (H, W) uint8
            image_label = self.gt[_index]        # (H, W) uint8 {0, 255}

            if offset // self.n_anom >= self.reuse_times:
                # Ramo copy_paste (augmentation): attivo solo se augmentors non è vuoto
                image, image_label = self.copy_paste(image, image_label)

            image       = Image.fromarray(image)
            image_label = Image.fromarray(image_label)
            label = 1
        else:
            image       = Image.fromarray(self.images[index])
            image_label = Image.fromarray(self.gt[index])
            label       = self.labels[index]

        # gt_label: (1, H, W) float — 1 canale, NO repeat(3,1,1) come nell'originale RGB
        gt_label = self.transform_mask(image_label)   # (1, H, W)

        return {
            "image":    self.transform(image),  # (1, H, W) float normalizzato
            "label":    label,
            "gt_label": gt_label,
        }

    def randAugmenter(self):
        """Seleziona 3 augmentor casuali dalla lista. No-op se augmentors è vuoto."""
        if len(self.augmentors) < 3:
            return A.Compose([])
        aug_ind = np.random.choice(np.arange(len(self.augmentors)), 3, replace=False)
        return A.Compose([self.augmentors[i] for i in aug_ind])

    def copy_paste(self, img: np.ndarray, mask: np.ndarray):
        """
        Incolla la regione anomala su un'immagine normale casuale.
        Per immagini grayscale (H,W): non c'è conversione BGR→RGB.
        """
        n_idx   = np.random.randint(self.n_norm)
        aug     = self.randAugmenter()
        n_image = self.images[self.labels == 0][n_idx].copy()  # (H, W)

        augmented   = aug(image=img, mask=mask)
        aug_image   = augmented["image"]
        aug_mask    = augmented["mask"]

        # Incolla la regione anomala sull'immagine normale
        n_image[aug_mask == 255] = aug_image[aug_mask == 255]
        return n_image, aug_mask


# -----------------------------------------------------------------------
# Verifica con dati dummy in-memory
# -----------------------------------------------------------------------
print("Verifica CustomGrayscaleAD con dati dummy...")

import tempfile, os
import torch

# Creiamo array dummy compatibili con il formato atteso
N_NORM, N_ANOM = 10, 3
H, W = 224, 224

rng = np.random.default_rng(0)
X_dummy  = rng.integers(0, 256, (N_NORM + N_ANOM, 1, H, W), dtype=np.uint8)
Y_dummy  = np.array([0]*N_NORM + [1]*N_ANOM, dtype=np.int64)
GT_dummy = np.zeros((N_NORM + N_ANOM, 1, H, W), dtype=np.uint8)
GT_dummy[N_NORM:, :, 100:200, 100:200] = 255  # maschere non vuote per le anomalie

with tempfile.TemporaryDirectory() as tmpdir:
    np.save(os.path.join(tmpdir, "X_train.npy"),  X_dummy)
    np.save(os.path.join(tmpdir, "Y_train.npy"),  Y_dummy)
    np.save(os.path.join(tmpdir, "GT_train.npy"), GT_dummy)
    np.save(os.path.join(tmpdir, "X_test.npy"),   X_dummy)
    np.save(os.path.join(tmpdir, "Y_test.npy"),   Y_dummy)
    np.save(os.path.join(tmpdir, "GT_test.npy"),  GT_dummy)

    ds_train = CustomGrayscaleAD(tmpdir, train=True)
    ds_test  = CustomGrayscaleAD(tmpdir, train=False)

    # Lunghezza attesa in train: n_norm + (repeat + reuse) * n_anom
    expected_train_len = N_NORM + (ds_train.repeat_time + ds_train.reuse_times) * N_ANOM
    assert len(ds_train) == expected_train_len, \
        f"Lunghezza train errata: {len(ds_train)} vs {expected_train_len}"

    # Campione normale
    sample_norm = ds_train[0]
    assert sample_norm["image"].shape    == (1, H, W), f"image shape: {sample_norm['image'].shape}"
    assert sample_norm["gt_label"].shape == (1, H, W), f"gt shape: {sample_norm['gt_label'].shape}"

    # Campione dal range di oversampling
    sample_over = ds_train[N_NORM + N_ANOM]
    assert sample_over["image"].shape    == (1, H, W)
    assert sample_over["gt_label"].shape == (1, H, W)

    # Range di normalizzazione
    img_min = sample_norm["image"].min().item()
    img_max = sample_norm["image"].max().item()
    assert img_min >= -1.1 and img_max <= 1.1, \
        f"Range normalizzazione fuori bounds: [{img_min:.2f}, {img_max:.2f}]"

    print(f"  ✓ len(train):        {len(ds_train)}  (atteso {expected_train_len})")
    print(f"  ✓ len(test):         {len(ds_test)}")
    print(f"  ✓ image shape:       {sample_norm['image'].shape}")
    print(f"  ✓ gt_label shape:    {sample_norm['gt_label'].shape}")
    print(f"  ✓ image range:       [{img_min:.3f}, {img_max:.3f}]")
    print(f"  ✓ dim:               {ds_train.dim}")
    print(f"  ✓ n_norm / n_anom:   {ds_train.n_norm} / {ds_train.n_anom}")

print("\n✓ CustomGrayscaleAD definita e verificata.")

In [ ]:
# TrainerGray + launch_gray ===

import os
import sys
import numpy as np
import torch
import torch.optim
from torch.utils.data import DataLoader
from tqdm import tqdm
from time import time
from pathlib import Path

# Import dalla repo (già in sys.path dalla cella 4)
from loss import AEXAD_loss_norm_vgg

# PRIMA (importa dalla repo — rotto su PyTorch recente)
#from trainer import BalancedBatchSampler

# DOPO (ridefinizione locale compatibile)
from torch.utils.data import Sampler

class BalancedBatchSampler(Sampler):
    def __init__(self, steps_per_epoch, batch_size, dataset):
        super().__init__()          # nessun argomento — fix per PyTorch >= 1.8
        self.dataset         = dataset
        self.steps_per_epoch = steps_per_epoch
        self.batch_size      = batch_size
        self.n_normal        = 2 * self.batch_size // 3
        self.n_anomaly       = self.batch_size - self.n_normal
        self.normal_generator  = self.randomGenerator(self.dataset.normal_idx)
        self.outlier_generator = self.randomGenerator(self.dataset.anomaly_idx)

    def randomGenerator(self, list):
        while True:
            for i in np.random.permutation(list):
                yield i

    def __len__(self):
        return self.steps_per_epoch

    def __iter__(self):
        for _ in range(self.steps_per_epoch):
            batch = []
            for _ in range(self.n_normal):
                batch.append(next(self.normal_generator))
            for _ in range(self.n_anomaly):
                batch.append(next(self.outlier_generator))
            yield batch

# -----------------------------------------------------------------------
# Perché NON monkey-patchare Trainer e NON copiare trainer.py?
#
# Trainer.__init__ fa due cose che ci creano problemi:
#   1. Istanzia modello e dataloader via string dispatch ('vgg_cnn', 'mvtec'...)
#      — noi abbiamo classi nuove che non rientrano in nessun branch
#   2. Il branch 'vgg_cnn' ha un assert isinstance(..., CustomVGGAD_CP)
#      — fallisce perché il nostro dataset è CustomGrayscaleAD
#
# Trainer.train() ha un problema aggiuntivo:
#   - Risolve il nome del file di salvataggio con isinstance(model, ResNet_CNN_Attn)
#     che è False per ResNet_CNN_Attn_Gray → UnboundLocalError su 'name'
#
# Soluzione più pulita: TrainerGray riceve modello e loaders già istanziati
# (dependency injection). Il codice di train() e test() è identico all'originale
# a parte le due righe problematiche. Nessun file della repo viene toccato.
# -----------------------------------------------------------------------

class TrainerGray:
    """
    Trainer per ResNet_CNN_Attn_Gray + CustomGrayscaleAD.
    Interfaccia identica a Trainer, ma accetta istanze già costruite
    invece di costruirle internamente via string dispatch.
    """

    def __init__(
        self,
        train_loader: DataLoader,
        test_loader:  DataLoader,
        model:        torch.nn.Module,
        criterion:    torch.nn.Module,
        use_cuda:     bool = True,
    ):
        self.train_loader = train_loader
        self.test_loader  = test_loader
        self.model        = model
        self.criterion    = criterion
        self.cuda         = use_cuda and torch.cuda.is_available()

        # Adam con lr=1e-3: identico all'originale.
        # Nota: solo i parametri con requires_grad=True vengono aggiornati.
        # Il frozen encoder (conv1, bn1, layer1, layer2) è escluso automaticamente.
        self.optimizer = torch.optim.Adam(
            filter(lambda p: p.requires_grad, self.model.parameters()),
            lr=1e-3
        )

        if self.cuda:
            self.model     = self.model.cuda()
            self.criterion = self.criterion.cuda()

        self.convergence = []  # lista di {'epoch': N, 'auc': X}, popolata da train()

    def train(self, epochs: int, save_path: str = ".", eval_every: int = 0) -> None:
        """
        Training loop identico a Trainer.train().
        Unica differenza: nome file checkpoint hardcoded come 'model_vgg_cnn_gray'
        (l'originale lo risolve con isinstance, che fallirebbe per la nostra classe).
        Se eval_every>0, valuta il test set ogni eval_every epoche e salva in
        self.convergence la pixel-AUC threshold-free (usa pixel_auc_mean_blurred
        dalla cella helper).
        """
        name = "model_vgg_cnn_gray"  # fix rispetto all'originale

        self.model.train()
        for epoch in range(epochs):
            train_loss   = 0.0
            train_loss_n = 0.0
            train_loss_a = 0.0
            ns = 0

            tbar = tqdm(self.train_loader, desc=f"Epoch {epoch}")
            for sample in tbar:
                image, label, gt_label = (
                    sample["image"],
                    sample["label"],
                    sample["gt_label"],
                )

                if self.cuda:
                    image    = image.cuda()
                    gt_label = gt_label.cuda()
                    label    = label.cuda()

                output = self.model(image)
                loss, loss_n, loss_a = self.criterion(output, image, gt_label, label)

                self.optimizer.zero_grad()
                loss.backward()
                self.optimizer.step()

                train_loss   += loss.item()
                train_loss_n += loss_n.item()
                train_loss_a += loss_a.item()
                ns += 1

                tbar.set_description(
                    f"Epoch {epoch} | loss {train_loss/ns:.4f} "
                    f"| loss_n {loss_n:.4f} | loss_a {loss_a:.4f}"
                )

            # --- Convergence eval (inline, senza checkpoint) ---
            if eval_every > 0 and (epoch + 1) % eval_every == 0:
                self.model.eval()
                htm, sc, gtm, lbl = self.test()
                auc = pixel_auc_mean_blurred(htm, gtm, lbl)
                self.convergence.append({"epoch": epoch + 1, "auc": float(auc)})
                print(f"  [conv] epoch {epoch+1}: pixel-AUC = {auc:.4f}")
                self.model.train()  # OBBLIGATORIO: ripristina training mode

        # Garanzia post-loop: BatchNorm/Dropout non devono restare in eval mode
        assert self.model.training, (
            "FAIL: model rimasto in eval mode dopo train() "
            "— controllare i branch eval_every"
        )

        torch.save(
            self.model.state_dict(),
            os.path.join(save_path, f"aexad_{name}.pt"),
        )

    def test(self):
        """
        Test loop identico a Trainer.test().
        Unica differenza: denormalizzazione sempre applicata
        (l'originale la condiziona con isinstance(..., CustomVGGAD_CP)).
        """
        heatmaps, scores, gtmaps, labels = [], [], [], []

        self.model.eval()
        with torch.no_grad():
            for sample in tqdm(self.test_loader, desc="Test"):
                image, label, gtmap = (
                    sample["image"],
                    sample["label"],
                    sample["gt_label"],
                )

                if self.cuda:
                    image = image.cuda()
                output = self.model(image).detach().cpu().numpy()
                image  = image.cpu().numpy()

                # Denormalizzazione: image era in [-1,1], riportiamo in [0,1]
                # image_denorm = image * std + mean
                mean = self.test_loader.dataset.mean[np.newaxis, :, np.newaxis, np.newaxis]
                std  = self.test_loader.dataset.std[np.newaxis,  :, np.newaxis, np.newaxis]
                image = image * std + mean

                # Heatmap grezza: errore di ricostruzione pixel per pixel
                heatmap = ((image - output) ** 2).astype(np.float32)
                score   = heatmap.reshape(image.shape[0], -1).mean(axis=-1)

                heatmaps.extend(heatmap)
                scores.extend(score)
                gtmaps.extend(gtmap.detach().numpy())
                labels.extend(label.detach().numpy())

        return (
            np.array(heatmaps),
            np.array(scores),
            np.array(gtmaps),
            np.array(labels),
        )


# -----------------------------------------------------------------------

def launch_gray(
    data_path:       str,
    epochs:          int,
    batch_size:      int,
    f,                         # funzione di trasformazione pixel anomali
    save_path:       str = ".",
    steps_per_epoch: int = 20,
    use_cuda:        bool = True,
    eval_every:      int = 0,  # 0=disattivato; N=valuta ogni N epoche
) -> tuple:
    """
    Entry point del training per il caso grayscale.
    Equivalente a launch() della repo, ma usa CustomGrayscaleAD
    e ResNet_CNN_Attn_Gray.

    Restituisce: heatmaps, scores, gtmaps, labels, tot_time, model, convergence
    """
    # --- Dataset ---
    ds_train = CustomGrayscaleAD(data_path, train=True,  img_size=224)
    ds_test  = CustomGrayscaleAD(data_path, train=False, img_size=224)

    # --- F4: Assert risoluzione (bug noto: risoluzione sbagliata senza errore) ---
    print(f"[F4] RESIZE_TO      = {RESIZE_TO}")
    print(f"[F4] img_size train  = {ds_train.img_size}")
    print(f"[F4] img_size test   = {ds_test.img_size}")
    assert RESIZE_TO == (224, 224), \
        f"FAIL F4: RESIZE_TO={RESIZE_TO}, atteso (224,224)"
    assert ds_train.img_size == 224, \
        f"FAIL F4: ds_train.img_size={ds_train.img_size}, atteso 224"
    assert ds_test.img_size == 224, \
        f"FAIL F4: ds_test.img_size={ds_test.img_size}, atteso 224"

        SANITY CHECK AUGMENTATION ===
    print(f"[ASSERT] N augmentor configurati: {len(ds_train.augmentors)}")
    print(f"[ASSERT] Tipi: {[type(a).__name__ for a in ds_train.augmentors]}")

    # --- Lambda_s: inferito dal dataset come nell'originale ---
    # lambda_s = N_campioni / N_anomalie
    # Rappresenta quanto peso dare alle anomalie rispetto ai normali
    # nella loss: con pochi anomalie il rapporto è alto, compensando
    # lo sbilanciamento del dataset.
    lambda_s = len(ds_train) / float(np.sum(ds_train.labels))
    print(f"lambda_s inferito: {lambda_s:.2f}")

    # --- DataLoader train: BalancedBatchSampler ---
    # Il sampler garantisce 2/3 normali e 1/3 anomalie per batch.
    # steps_per_epoch=20: numero di batch per epoca.
    sampler      = BalancedBatchSampler(steps_per_epoch, batch_size, ds_train)
    train_loader = DataLoader(ds_train, batch_sampler=sampler)

    # --- DataLoader test: sequenziale, batch_size=1 ---
    # batch_size=1 nel test semplifica la gestione delle heatmaps:
    # ogni output corrisponde esattamente a un'immagine.
    test_loader = DataLoader(ds_test, batch_size=1, shuffle=False)

    # --- Modello ---
    model = ResNet_CNN_Attn_Gray(dim=[1])

    # --- Loss ---
    # AEXAD_loss_norm_vgg richiede mean e std come np.array float32.
    # Usiamo float32 esplicitamente per evitare type mismatch con i
    # tensori float32 del modello: torch.from_numpy(float64) crea un
    # tensor float64, e float32 * float64 causa RuntimeError in PyTorch.
    mean_f32 = ds_train.mean.astype(np.float32)
    std_f32  = ds_train.std.astype(np.float32)

    criterion = AEXAD_loss_norm_vgg(
        mean=mean_f32,
        std=std_f32,
        lambda_p=None,    # inferito pixel per pixel dalla maschera
        lambda_s=lambda_s,
        f=f,
        cuda=use_cuda and torch.cuda.is_available(),
    )

    # --- Trainer ---
    trainer = TrainerGray(
        train_loader=train_loader,
        test_loader=test_loader,
        model=model,
        criterion=criterion,
        use_cuda=use_cuda,
    )

    # --- Training ---
    Path(save_path).mkdir(parents=True, exist_ok=True)
    t0 = time()
    trainer.train(epochs=epochs, save_path=save_path, eval_every=eval_every)
    tot_time = time() - t0

    # --- Test ---
    heatmaps, scores, gtmaps, labels = trainer.test()

    return heatmaps, scores, gtmaps, labels, tot_time, trainer.model, trainer.convergence


# -----------------------------------------------------------------------
# Smoke test: 2 epoche su dati dummy per verificare che tutto il pipeline
# giri senza errori prima di investire ore di training reale.
# -----------------------------------------------------------------------
print("Smoke test (2 epoche su dati dummy)...")

import tempfile

N_NORM, N_ANOM = 10, 3
H, W = 224, 224
rng  = np.random.default_rng(42)

X_d  = rng.integers(0, 256, (N_NORM + N_ANOM, 1, H, W), dtype=np.uint8)
Y_d  = np.array([0]*N_NORM + [1]*N_ANOM, dtype=np.int64)
GT_d = np.zeros_like(X_d)
GT_d[N_NORM:, :, 150:200, 150:200] = 255

with tempfile.TemporaryDirectory() as tmp:
    for split in ("train", "test"):
        np.save(os.path.join(tmp, f"X_{split}.npy"),  X_d)
        np.save(os.path.join(tmp, f"Y_{split}.npy"),  Y_d)
        np.save(os.path.join(tmp, f"GT_{split}.npy"), GT_d)

    with tempfile.TemporaryDirectory() as save_tmp:
        heatmaps, scores, gtmaps, labels, tot_time, model, convergence = launch_gray(
            data_path=tmp,
            epochs=2,
            batch_size=4,
            f=lambda x: torch.full_like(x, 2.0),
            save_path=save_tmp,
            steps_per_epoch=3,
            use_cuda=torch.cuda.is_available(),
        )

        assert heatmaps.shape[0] == N_NORM + N_ANOM, \
            f"heatmaps n campioni errato: {heatmaps.shape[0]}"
        assert heatmaps.shape[1:] == (1, H, W), \
            f"heatmaps shape errata: {heatmaps.shape}"
        assert scores.shape  == (N_NORM + N_ANOM,)
        assert gtmaps.shape  == (N_NORM + N_ANOM, 1, H, W)
        assert labels.shape  == (N_NORM + N_ANOM,)
        assert (Path(save_tmp) / "aexad_model_vgg_cnn_gray.pt").exists(), \
            "Checkpoint non salvato"

print(f"\n  ✓ heatmaps shape: {heatmaps.shape}")
print(f"  ✓ scores shape:   {scores.shape}")
print(f"  ✓ gtmaps shape:   {gtmaps.shape}")
print(f"  ✓ labels shape:   {labels.shape}")
print(f"  ✓ tempo smoke:    {tot_time:.1f}s")
print(f"  ✓ checkpoint:     aexad_model_vgg_cnn_gray.pt")
print("\n✓ launch_gray e TrainerGray definiti e verificati.")

In [ ]:
# blur adattivo + pixel_auc_mean_blurred (convergence + CELLA 21) ===
#
# UNICA FONTE per get_filter_sigma e blur_one.
# TrainerGray.train() usa pixel_auc_mean_blurred per le eval inline.
# CELLA 21 usa get_filter_sigma e blur_one invece di ridefinirli.

import numpy as np
from scipy.ndimage import gaussian_filter
from sklearn.metrics import roc_auc_score


def get_filter_sigma(ht_2d: np.ndarray, thres: float = 0.5, scale: float = 0.25) -> float:
    ht_norm = ht_2d / (ht_2d.max() + 1e-8)
    binht   = (ht_norm >= thres).astype(np.uint8)

    def rad_from_lines(gt):
        nc    = gt.shape[1]
        gtrow = gt.cumsum(axis=1) * gt
        drow  = gtrow[np.where(gtrow[:, :nc-1] > gtrow[:, 1:nc])]
        last_col = gtrow[gtrow[:, nc-1] > 0, nc-1]
        drow  = np.concatenate([drow, last_col])
        drow  = drow[drow > 3]
        rrow  = 0.5 * drow.mean() if drow.shape[0] > 0 else 0

        nr    = gt.shape[0]
        gtcol = gt.cumsum(axis=0) * gt
        dcol  = gtcol[np.where(gtcol[:nr-1, :] > gtcol[1:nr, :])]
        last_row = gtcol[nr-1, gtcol[nr-1, :] > 0]
        dcol  = np.concatenate([dcol, last_row])
        dcol  = dcol[dcol > 3]
        rcol  = 0.5 * dcol.mean() if dcol.shape[0] > 0 else 0
        return max(rcol, rrow)

    rad = rad_from_lines(binht)
    return max(rad * scale, 2.0)


def blur_one(ht2d: np.ndarray) -> np.ndarray:
    """Blur gaussiano adattivo su una singola heatmap 2D (H, W)."""
    return gaussian_filter(ht2d, sigma=get_filter_sigma(ht2d))


def pixel_auc_mean_blurred(
    heatmaps: np.ndarray,   # (N, 1, H, W)
    gtmaps:   np.ndarray,   # (N, 1, H, W)
    labels:   np.ndarray,   # (N,)
) -> float:
    """
    Pixel-AUC media sulle sole immagini anomale (label==1), dopo blur adattivo.
    Threshold-free: roc_auc_score su heatmap continua. NESSUNA binarizzazione.
    """
    anom_idx = np.where(np.asarray(labels) == 1)[0]
    aucs = []
    for i in anom_idx:
        gt_flat = (gtmaps[i, 0] > 0).astype(np.uint8).flatten()
        ht_flat = blur_one(heatmaps[i, 0]).flatten()
        try:
            auc = roc_auc_score(gt_flat, ht_flat)
        except ValueError:
            auc = float("nan")
        aucs.append(auc)
    return float(np.nanmean(aucs)) if aucs else float("nan")


print("✓ get_filter_sigma, blur_one, pixel_auc_mean_blurred definiti (helper condiviso).")

In [ ]:
# Generazione array NumPy per il training ===

import numpy as np
from pathlib import Path
from PIL import Image

def extract_dataset_gray(
    prepared_dir: Path,
    n_anom_per_cls: int,
    seed: int,
) -> tuple:
    """
    Costruisce gli array NumPy per il training a partire da busi_prepared/.

    Struttura attesa in prepared_dir:
        train/normal/           — immagini normali di training (no maschere)
        test/normal/            — immagini normali di test (no maschere)
        test/benign/images/     — immagini anomale benign
        test/benign/masks/      — maschere corrispondenti
        test/malignant/images/  — immagini anomale malignant
        test/malignant/masks/   — maschere corrispondenti

    Logica dello scenario "standard" del paper:
        - Training set  = tutti i normali di train
                        + n_anom_per_cls anomalie benign
                        + n_anom_per_cls anomalie malignant
        - Test set      = tutti i normali di test
                        + tutte le anomalie benign NON usate nel training
                        + tutte le anomalie malignant NON usate nel training

    Output:
        X_train, Y_train, X_test, Y_test, GT_train, GT_test — array NumPy
        test_anom_file_list — lista di dict {"cls": str, "fname": str}
                              nell'ordine esatto in cui le anomalie
                              compaiono in X_test. Serve alla cella 23
                              per risalire al file originale dato un indice.
    """
    rng = np.random.default_rng(seed)

    X_train, Y_train, GT_train = [], [], []
    X_test,  Y_test,  GT_test  = [], [], []

    # Lista file anomali nel test set, popolata in parallelo al loop.
    # Perché costruirla qui dentro e non dopo con un secondo rng?
    # Se creassimo un secondo generatore con lo stesso seed fuori da questa
    # funzione, qualsiasi modifica futura al codice potrebbe desincronizzarlo
    # silenziosamente. Costruirla dentro il loop garantisce che l'ordine sia
    # identico a X_test per costruzione, non per coincidenza.
    test_anom_file_list = []

    def load_gray_chw(img_path: Path) -> np.ndarray:
        """Carica immagine grayscale e la restituisce come (1, H, W) uint8."""
        arr = np.array(Image.open(img_path).convert("L"), dtype=np.uint8)
        return arr[np.newaxis, :, :]

    def load_mask_chw(mask_path: Path) -> np.ndarray:
        """Carica maschera e la restituisce come (1, H, W) uint8 con valori {0,255}."""
        arr = np.array(Image.open(mask_path).convert("L"), dtype=np.uint8)
        return arr[np.newaxis, :, :]

    # --- Normali di training ---
    # sorted() garantisce ordine deterministico cross-platform:
    # glob() non garantisce ordine stabile su tutti i filesystem.
    train_normal_files = sorted((prepared_dir / "train" / "normal").glob("*.png"))
    for img_path in train_normal_files:
        x = load_gray_chw(img_path)
        X_train.append(x)
        GT_train.append(np.zeros_like(x, dtype=np.uint8))
        Y_train.append(0)
    print(f"Normali train:    {len(train_normal_files)}")

    # --- Anomalie: selezione train/test per classe ---
    # Per ogni classe anomala:
    #   - Mescoliamo i file con rng (seed-specifico) → riproducibilità garantita
    #   - Primi n_anom_per_cls → training set
    #   - Restanti → test set
    # Le anomalie vengono estratte da test/ perché è lì che le abbiamo
    # messe tutte in cella 10. La divisione train/test avviene qui,
    # al momento della generazione degli .npy, con il seed del run corrente.
    for cls in ("benign", "malignant"):
        img_dir  = prepared_dir / "test" / cls / "images"
        mask_dir = prepared_dir / "test" / cls / "masks"

        all_imgs = sorted(img_dir.glob("*.png"))
        n_total  = len(all_imgs)

        if n_total < n_anom_per_cls:
            raise ValueError(
                f"{cls}: richieste {n_anom_per_cls} anomalie per training, "
                f"disponibili solo {n_total}"
            )

        shuffled   = [all_imgs[i] for i in rng.permutation(n_total)]
        train_imgs = shuffled[:n_anom_per_cls]
        test_imgs  = shuffled[n_anom_per_cls:]

        for img_path in train_imgs:
            mask_path = mask_dir / img_path.name
            x  = load_gray_chw(img_path)
            gt = load_mask_chw(mask_path)
            X_train.append(x)
            GT_train.append(gt)
            Y_train.append(1)

        for img_path in test_imgs:
            mask_path = mask_dir / img_path.name
            x  = load_gray_chw(img_path)
            gt = load_mask_chw(mask_path)
            X_test.append(x)
            GT_test.append(gt)
            Y_test.append(1)
            # Registriamo classe e nome file nell'ordine esatto di inserimento
            # in X_test. È questo il mapping indice → file che serve alla
            # cella 23 per mostrare l'immagine giusta accanto alla heatmap.
            # Seed diversi producono shuffle diversi → ordini diversi →
            # json diversi. Per questo il json è seed-specifico.
            test_anom_file_list.append({"cls": cls, "fname": img_path.name})

        print(f"{cls:12s}: {n_anom_per_cls} train, {len(test_imgs)} test")

    # --- Normali di test ---
    # Aggiunti dopo le anomalie: così Y_test ha prima tutti gli 1
    # e poi tutti gli 0, coerente con come l'originale costruisce il test set.
    test_normal_files = sorted((prepared_dir / "test" / "normal").glob("*.png"))
    for img_path in test_normal_files:
        x = load_gray_chw(img_path)
        X_test.append(x)
        GT_test.append(np.zeros_like(x, dtype=np.uint8))
        Y_test.append(0)
    print(f"Normali test:     {len(test_normal_files)}")

    # --- Assemblaggio array finali ---
    X_train  = np.array(X_train,  dtype=np.uint8)
    Y_train  = np.array(Y_train,  dtype=np.int64)
    GT_train = np.array(GT_train, dtype=np.uint8)
    X_test   = np.array(X_test,   dtype=np.uint8)
    Y_test   = np.array(Y_test,   dtype=np.int64)
    GT_test  = np.array(GT_test,  dtype=np.uint8)

    print(f"\nX_train:  {X_train.shape}  Y_train:  {Y_train.shape}  "
          f"(anomalie: {int(Y_train.sum())})")
    print(f"X_test:   {X_test.shape}   Y_test:   {Y_test.shape}   "
          f"(anomalie: {int(Y_test.sum())})")
    print(f"GT_train: {GT_train.shape}")
    print(f"GT_test:  {GT_test.shape}")

    return X_train, Y_train, X_test, Y_test, GT_train, GT_test, test_anom_file_list


def save_npy_dataset(
    out_dir: Path,
    X_train, Y_train, X_test, Y_test, GT_train, GT_test,
) -> None:
    """
    Salva i sei array in out_dir come file .npy.
    Idempotente: sovrascrive se già esistono.
    """
    out_dir.mkdir(parents=True, exist_ok=True)
    np.save(out_dir / "X_train.npy",  X_train)
    np.save(out_dir / "Y_train.npy",  Y_train)
    np.save(out_dir / "X_test.npy",   X_test)
    np.save(out_dir / "Y_test.npy",   Y_test)
    np.save(out_dir / "GT_train.npy", GT_train)
    np.save(out_dir / "GT_test.npy",  GT_test)
    total_mb = sum(
        (out_dir / f"{n}.npy").stat().st_size
        for n in ("X_train","Y_train","X_test","Y_test","GT_train","GT_test")
    ) / 1e6
    print(f"✓ .npy salvati in {out_dir}  ({total_mb:.1f} MB totali)")


# --- Verifica dry-run ---
print("Verifica extract_dataset_gray su busi_prepared/ reale...")
_n_anom_dry = N_ANOM_PER_CLS if 'N_ANOM_PER_CLS' in globals() else 5
print(f"(usa seed=101, n_anom_per_cls={_n_anom_dry})\n")

_X_tr, _Y_tr, _X_te, _Y_te, _GT_tr, _GT_te, _tafl = extract_dataset_gray(
    prepared_dir=OUT_DIR,
    n_anom_per_cls=_n_anom_dry,
    seed=101,
)

assert _X_tr.ndim  == 4 and _X_tr.shape[1]  == 1, f"X_train shape: {_X_tr.shape}"
assert _GT_tr.ndim == 4 and _GT_tr.shape[1] == 1, f"GT_train shape: {_GT_tr.shape}"
assert _X_tr.shape[0]  == _Y_tr.shape[0],  "X/Y train mismatch"
assert _X_te.shape[0]  == _Y_te.shape[0],  "X/Y test mismatch"
assert _X_tr.shape[2:] == (224, 224),       f"Risoluzione train errata: {_X_tr.shape}"
assert int(_Y_tr.sum()) == _n_anom_dry * 2, f"Anomalie train attese {_n_anom_dry * 2}, trovate {int(_Y_tr.sum())}"
assert len(_tafl) == int(_Y_te.sum()), \
    f"test_anom_file_list lunghezza errata: {len(_tafl)} vs {int(_Y_te.sum())}"
assert all("cls" in e and "fname" in e for e in _tafl), \
    "Elementi malformati in test_anom_file_list"

print(f"\n✓ test_anom_file_list: {len(_tafl)} entry  (es: {_tafl[0]})")
print("✓ Tutti i controlli superati.")
print("✓ Funzioni extract_dataset_gray e save_npy_dataset pronte.")

In [ ]:
# Seed deterministici ===

import random
import numpy as np
import torch

def set_all_seeds(seed: int) -> None:
    """
    Imposta seed identici su tutti i generatori di numeri casuali
    coinvolti nel training.

    Perché coprire tutti e quattro?
    PyTorch, NumPy, Python random e cuDNN sono generatori indipendenti:
    fissare solo torch.manual_seed() lascia NumPy e Python liberi di
    produrre sequenze diverse tra un run e l'altro. Basta una chiamata
    a np.random o random.choice in qualsiasi punto del codice (dataloader,
    augmentation, sampler) per rendere il run non riproducibile.

    Perché torch.backends.cudnn.deterministic = True?
    Alcune operazioni di cuDNN (convoluzione, pooling) hanno implementazioni
    non deterministiche per default: scelgono l'algoritmo più veloce per
    l'hardware corrente, e algoritmi diversi producono risultati leggermente
    diversi per via dell'aritmetica floating point. Con deterministic=True
    cuDNN usa sempre lo stesso algoritmo, a costo di un leggero rallentamento.

    Perché torch.backends.cudnn.benchmark = False?
    benchmark=True fa eseguire a cuDNN una fase di warm-up in cui prova
    più algoritmi e sceglie il più veloce per il profilo di input corrente.
    Questa fase è non deterministica. Va disattivata per la riproducibilità.
    """
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)          # copre tutti i device GPU
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark     = False

# --- Verifica ---
set_all_seeds(101)

# Verifichiamo che due chiamate consecutive con lo stesso seed
# producano la stessa sequenza random in tutti e tre i generatori.
r1_np  = np.random.randint(0, 10000)
r1_pt  = torch.randint(0, 10000, (1,)).item()
r1_py  = random.randint(0, 10000)

set_all_seeds(101)

r2_np  = np.random.randint(0, 10000)
r2_pt  = torch.randint(0, 10000, (1,)).item()
r2_py  = random.randint(0, 10000)

assert r1_np == r2_np, f"NumPy non deterministico: {r1_np} != {r2_np}"
assert r1_pt == r2_pt, f"PyTorch non deterministico: {r1_pt} != {r2_pt}"
assert r1_py == r2_py, f"Python random non deterministico: {r1_py} != {r2_py}"

print(f"✓ NumPy seed:        {r1_np} == {r2_np}")
print(f"✓ PyTorch seed:      {r1_pt} == {r2_pt}")
print(f"✓ Python random:     {r1_py} == {r2_py}")
print(f"✓ cuDNN deterministic: {torch.backends.cudnn.deterministic}")
print(f"✓ cuDNN benchmark:     {torch.backends.cudnn.benchmark}")
print("\n✓ set_all_seeds definita e verificata.")

In [ ]:
# # Pulizia pre-run ===
# import shutil
# from pathlib import Path

# SEEDS_ALL = [101, 237, 359, 461, 491]

# # --- Sposta risultati vecchi ---
# runs_dir   = PROJECT_ROOT / "runs"
# backup_dir = PROJECT_ROOT / "runs_5anom_aug"  # ← aggiorna il nome ogni volta
# backup_dir.mkdir(parents=True, exist_ok=True)

# for seed in SEEDS_ALL:
#     src = runs_dir / f"seed{seed}"
#     dst = backup_dir / f"seed{seed}"
#     if src.exists() and not dst.exists():
#         shutil.move(str(src), str(dst))
#         print(f"  ✓ seed {seed}: spostato in {dst}")

# for fname in ("results.json", "results.csv", "qualitative.png"):
#     src = runs_dir / fname
#     if src.exists():
#         shutil.move(str(src), str(backup_dir / fname))

# # --- Cancella .npy locali ---
# for seed in SEEDS_ALL:
#     npy_dir = PROJECT_ROOT / "npy" / f"busi_seed{seed}"   # ← PATH LOCALE CORRETTO
#     if npy_dir.exists():
#         shutil.rmtree(npy_dir)
#         print(f"  ✓ {npy_dir.name} rimossa")

# print("\n✓ Pulizia completata.")


In [ ]:
# Parametri del run ===

import torch
from pathlib import Path

SEEDS           = [101, 237, 359, 461, 491]
N_ANOM_PER_CLS  = 10
EPOCHS          = 100
BATCH_SIZE      = 32
STEPS_PER_EPOCH = 20

# --- Modalità di valutazione ---
# EVAL_EVERY controlla OGNI QUANTE epoche si valuta il test set durante il training.
#   EVAL_EVERY = 50      -> valuta a 50,100,150,200,250  -> CURVA DI CONVERGENZA
#   EVAL_EVERY = EPOCHS  -> valuta solo all'ultima epoca -> RUN LISCIA (solo AUC finale)
# NON è il numero di epoche di training: quello è EPOCHS. Qui scegli solo la frequenza di eval.
#EVAL_EVERY      = 50          # convergenza
EVAL_EVERY    = 0      # <-- decommenta QUESTA (e commenta quella sopra) per run liscia

F_ANOMALY = lambda x: torch.full_like(x, 2.0)

# F2: valore reale dal lambda, nessuna stringa hardcoded
_probe = torch.zeros(1)
F_ANOMALY_VALUE = float(F_ANOMALY(_probe).item())
assert F_ANOMALY_VALUE == 2.0, f"FAIL F2: {F_ANOMALY_VALUE}"

NPY_BASE  = PROJECT_ROOT

# --- Cartella output: UNA PER ESPERIMENTO ---
# Cambiare ad ogni esperimento diverso, altrimenti il resume in CELLA 19 salta
# i seed già presenti e ti restituisce i risultati di un'altra run.
#   "runs_conv"      -> convergenza baseline senza layer (NON riusare)
#   "runs_layer224"  -> variante con layer di compressione
RUN_NAME  = "runs_224_newLayer_dense"       # <-- IMPOSTARE il nome giusto per QUESTO esperimento
RUNS_BASE = DRIVE_BASE / RUN_NAME
RUNS_DIR  = RUNS_BASE

# Coerenza: se valuti solo alla fine, l'etichetta lo dice
#EVAL_MODE = "convergenza" if EVAL_EVERY < EPOCHS else "liscia (solo finale)"
EVAL_MODE = "convergenza" if EVAL_EVERY > 0 else "liscia (solo finale)"


print("Parametri esperimento:")
print(f"  RUN_NAME:        {RUN_NAME}")
print(f"  SEEDS:           {SEEDS}")
print(f"  N_ANOM_PER_CLS:  {N_ANOM_PER_CLS}  (tot: {N_ANOM_PER_CLS*2})")
print(f"  EPOCHS:          {EPOCHS}")
print(f"  EVAL_EVERY:      {EVAL_EVERY}  -> modalità: {EVAL_MODE}")
print(f"  BATCH_SIZE:      {BATCH_SIZE}")
print(f"  STEPS_PER_EPOCH: {STEPS_PER_EPOCH}")
print(f"  F_ANOMALY_VALUE: {F_ANOMALY_VALUE}")
print(f"\n[F2] f_anomaly_value = {F_ANOMALY_VALUE}  ✓")
print(f"  NPY_BASE:        {NPY_BASE}")
print(f"  RUNS_BASE:       {RUNS_BASE}")

In [ ]:
# Esecuzione training (loop su tutti i SEEDS con resume) ===

import json
import shutil
import traceback
from pathlib import Path
from time import time

print(f"\n{'#'*70}")
print(f"# Avvio esperimento: {len(SEEDS)} seed da eseguire")
print(f"# SEEDS = {SEEDS}")
print(f"{'#'*70}\n")

t_total_start = time()

for seed_loop_idx, SEED in enumerate(SEEDS, start=1):
    print(f"\n{'='*70}")
    print(f"SEED {SEED}  ({seed_loop_idx}/{len(SEEDS)})")
    print(f"{'='*70}")

    # Calcolo path per-seed
    NPY_DIR = NPY_BASE / f"busi_seed{SEED}"
    OUT_RUN = RUNS_BASE / f"seed{SEED}"

    # --- RESUME CHECK ---
    # Considera "completato" un seed che ha tutti i file output principali.
    # Se uno solo manca, rilancia da zero il training di quel seed.
    required_files = [
        "heatmaps.npy", "scores.npy", "gtmaps.npy", "labels.npy",
        "config.json", "test_anom_files.json", "convergence.json",
    ]
    has_all_outputs = all((OUT_RUN / f).exists() for f in required_files)
    has_checkpoint  = OUT_RUN.exists() and len(list(OUT_RUN.glob("*.pt"))) >= 1
    has_no_error    = not (OUT_RUN / "error.txt").exists()

    if has_all_outputs and has_checkpoint and has_no_error:
        # Se serve, leggi config per stampare un riepilogo
        try:
            cfg = json.loads((OUT_RUN / "config.json").read_text())
            t_train_prev = cfg.get("train_time_s", "?")
            print(f"✓ Seed {SEED} GIÀ COMPLETATO  (train_time={t_train_prev}s)")
            print(f"  → skip, output in {OUT_RUN}")
        except Exception:
            print(f"✓ Seed {SEED} già completato, skip.")
        continue

    # Se OUT_RUN esiste ma è incompleto, lo segnalo
    if OUT_RUN.exists() and not has_all_outputs:
        print(f"⚠ Seed {SEED}: cartella esistente ma incompleta, rilancio training.")

    OUT_RUN.mkdir(parents=True, exist_ok=True)
    error_path = OUT_RUN / "error.txt"

    try:
        # --- 1. Seed ---
        set_all_seeds(SEED)
        print(f"✓ Seed impostato: {SEED}")

        # --- 2. Generazione .npy ---
        npy_files_check = ["X_train","Y_train","X_test","Y_test","GT_train","GT_test"]
        npy_ok    = all((NPY_DIR / f"{f}.npy").exists() for f in npy_files_check)
        json_ok   = (NPY_DIR / "test_anom_files.json").exists()

        if npy_ok and json_ok:
            print(f"✓ .npy e test_anom_files.json già presenti in {NPY_DIR}, skip.")
        else:
            print("Generazione .npy...")
            t_npy = time()
            X_tr, Y_tr, X_te, Y_te, GT_tr, GT_te, test_anom_file_list = extract_dataset_gray(
                prepared_dir=OUT_DIR,
                n_anom_per_cls=N_ANOM_PER_CLS,
                seed=SEED,
            )
            save_npy_dataset(NPY_DIR, X_tr, Y_tr, X_te, Y_te, GT_tr, GT_te)

            (NPY_DIR / "test_anom_files.json").write_text(
                json.dumps(test_anom_file_list)
            )
            print(f"✓ test_anom_files.json salvato ({len(test_anom_file_list)} entry)")
            print(f"✓ .npy generati in {time()-t_npy:.1f}s")

        # --- 3. Training ---
        print(f"\nAvvio training — seed {SEED}, {EPOCHS} epoche...")
        t_train = time()

        heatmaps, scores, gtmaps, labels, tot_time, model, convergence = launch_gray(
            data_path       = str(NPY_DIR),
            epochs          = EPOCHS,
            batch_size      = BATCH_SIZE,
            f               = F_ANOMALY,
            save_path       = str(OUT_RUN),
            steps_per_epoch = STEPS_PER_EPOCH,
            use_cuda        = True,
            eval_every      = EVAL_EVERY,
        )

        print(f"\n✓ Training completato in {tot_time/60:.1f} min")

        # --- 3b. Salvataggio convergence.json ---
        (OUT_RUN / "convergence.json").write_text(json.dumps(convergence, indent=2))
        print(f"✓ convergence.json salvato ({len(convergence)} checkpoint)")

        # --- 4. Salvataggio risultati ---
        print("Salvataggio risultati...")

        np.save(OUT_RUN / "heatmaps.npy", heatmaps.astype(np.float32))
        np.save(OUT_RUN / "scores.npy",   scores)
        np.save(OUT_RUN / "gtmaps.npy",   gtmaps)
        np.save(OUT_RUN / "labels.npy",   labels)

        shutil.copy(
            NPY_DIR / "test_anom_files.json",
            OUT_RUN / "test_anom_files.json",
        )

        config = {
            "seed":             SEED,
            "n_anom_per_cls":   N_ANOM_PER_CLS,
            "epochs":           EPOCHS,
            "eval_every":       EVAL_EVERY,
            "batch_size":       BATCH_SIZE,
            "steps_per_epoch":  STEPS_PER_EPOCH,
            "f_anomaly_value":  F_ANOMALY_VALUE,   # F2: valore reale dal lambda
            "resize":           list(RESIZE_TO),
            "mean":             0.0,
            "std":              1.0,
            "architecture":     "ResNet_CNN_Attn_Gray",
            "augmentation":     "MultiplicativeNoise+HorizontalFlip+blur+SSR+elastic+CLAHE",
            "train_time_s":     round(tot_time, 1),
        }
        (OUT_RUN / "config.json").write_text(json.dumps(config, indent=2))
        (OUT_RUN / "train_time.txt").write_text(f"{tot_time:.1f} secondi\n")

        pt_files = list(OUT_RUN.glob("*.pt"))
        assert len(pt_files) == 1, f"Checkpoint .pt non trovato in {OUT_RUN}"

        if error_path.exists():
            error_path.unlink()

        print(f"✓ heatmaps.npy  {heatmaps.shape}")
        print(f"✓ scores.npy    {scores.shape}")
        print(f"✓ gtmaps.npy    {gtmaps.shape}")
        print(f"✓ labels.npy    {labels.shape}")
        print(f"✓ config.json + train_time.txt + checkpoint .pt")
        print(f"\n✓ Seed {SEED} completato in {tot_time/60:.1f} min — output in {OUT_RUN}")

    except Exception as e:
        tb = traceback.format_exc()
        error_path.write_text(f"SEED={SEED}\n\n{tb}")
        print(f"\n✗ Errore seed {SEED}, salvato in {error_path}")
        print(f"  Continuo con i seed successivi...")
        # NON faccio raise: voglio che il loop continui anche se un seed fallisce
        continue

# Riepilogo finale
t_total = time() - t_total_start
print(f"\n{'#'*70}")
print(f"# ESPERIMENTO COMPLETATO  (tempo totale: {t_total/60:.1f} min)")
print(f"{'#'*70}")

# Stampa stato finale di ogni seed
print("\nStato finale:")
for SEED in SEEDS:
    OUT_RUN = RUNS_BASE / f"seed{SEED}"
    if (OUT_RUN / "heatmaps.npy").exists() and not (OUT_RUN / "error.txt").exists():
        cfg = json.loads((OUT_RUN / "config.json").read_text())
        print(f"  Seed {SEED}: ✓ OK  ({cfg.get('train_time_s', '?')}s)")
    elif (OUT_RUN / "error.txt").exists():
        print(f"  Seed {SEED}: ✗ ERRORE  (vedi {OUT_RUN}/error.txt)")
    else:
        print(f"  Seed {SEED}: ? incompleto/mancante")

In [ ]:
# Check post-run ===

import json
import numpy as np
from pathlib import Path

print(f"{'='*55}")
print(f"CHECK POST-RUN — seed {SEED}")
print(f"{'='*55}\n")

# --- 1. File presenti in OUT_RUN ---
print("File in OUT_RUN:")
expected_files = [
    "heatmaps.npy", "scores.npy", "gtmaps.npy", "labels.npy",
    "config.json", "train_time.txt", "aexad_model_vgg_cnn_gray.pt",
]
for fname in expected_files:
    fpath = OUT_RUN / fname
    exists = fpath.exists()
    size   = f"{fpath.stat().st_size / 1e6:.1f} MB" if exists else "MANCANTE"
    marker = "✓" if exists else "✗"
    print(f"  {marker} {fname:<35} {size}")

missing = [f for f in expected_files if not (OUT_RUN / f).exists()]
assert not missing, f"File mancanti: {missing}"

# --- 2. Shape degli array ---
print("\nShape array:")
arrays = {}
for name in ("heatmaps", "scores", "gtmaps", "labels"):
    arr = np.load(OUT_RUN / f"{name}.npy")
    arrays[name] = arr
    print(f"  {name:<12} {str(arr.shape):<25} dtype={arr.dtype}")

n_test = arrays["labels"].shape[0]
assert arrays["heatmaps"].shape == (n_test, 1, 224, 224), \
    f"heatmaps shape errata: {arrays['heatmaps'].shape}"
assert arrays["gtmaps"].shape   == (n_test, 1, 224, 224), \
    f"gtmaps shape errata: {arrays['gtmaps'].shape}"
assert arrays["scores"].shape   == (n_test,)
assert arrays["labels"].shape   == (n_test,)

# --- 3. Distribuzione labels ---
n_anom = int(arrays["labels"].sum())
n_norm = n_test - n_anom
print(f"\nDistribuzione test set:")
print(f"  normali:  {n_norm}")
print(f"  anomalie: {n_anom}")
print(f"  totale:   {n_test}")

# --- 4. Sanity check heatmaps ---
# Le heatmaps sono errori di ricostruzione (input - output)²: sempre >= 0.
# Se ci sono valori negativi c'è un bug nella denormalizzazione.
assert arrays["heatmaps"].min() >= 0, \
    f"Heatmap con valori negativi: min={arrays['heatmaps'].min():.4f}"

# Score medio delle anomalie deve essere > score medio dei normali:
# se il modello ha imparato qualcosa, ricostruisce male le anomalie.
mean_score_anom = arrays["scores"][arrays["labels"] == 1].mean()
mean_score_norm = arrays["scores"][arrays["labels"] == 0].mean()
print(f"\nScore medio anomalie: {mean_score_anom:.4f}")
print(f"Score medio normali:  {mean_score_norm:.4f}")
separation = mean_score_anom > mean_score_norm
print(f"Separazione (anom > norm): {'✓ SÌ' if separation else '✗ NO — il modello non discrimina'}")

# --- 5. Config ---
print("\nconfig.json:")
config = json.loads((OUT_RUN / "config.json").read_text())
for k, v in config.items():
    print(f"  {k:<20} {v}")

# --- 6. Tempo training ---
train_time = (OUT_RUN / "train_time.txt").read_text().strip()
print(f"\nTempo training: {train_time}")

print(f"\n✓ Check post-run completato — seed {SEED}")

In [ ]:
# Post-processing e metriche per immagine ===

import sys
import json
import numpy as np
from pathlib import Path
from scipy.ndimage import gaussian_filter
from sklearn.metrics import roc_auc_score, f1_score

sys.path.insert(0, str(PROJECT_ROOT))
from tools.evaluation_metrics import Xauc, IoU

# -----------------------------------------------------------------------
# POST-PROCESSING
# -----------------------------------------------------------------------

# get_filter_sigma e blur_one sono definiti nella cella helper (unica fonte).

def binarize_heatmap(ht_2d: np.ndarray, k: float = 1.0) -> np.ndarray:
    thr = ht_2d.mean() + k * ht_2d.std()
    return (ht_2d >= thr).astype(np.uint8)


def compute_metrics_single(
    heatmap_2d: np.ndarray,
    gt_2d:      np.ndarray,
) -> dict:
    gt_bin  = (gt_2d > 0).astype(np.uint8).flatten()
    ht_flat = heatmap_2d.flatten()
    ht_bin  = binarize_heatmap(heatmap_2d).flatten()

    try:
        auc = roc_auc_score(gt_bin, ht_flat)
    except ValueError:
        auc = float("nan")

    if ht_bin.sum() == 0 or gt_bin.sum() == 0:
        f1 = 0.0
    else:
        f1 = f1_score(gt_bin, ht_bin, zero_division=0)

    if ht_bin.sum() == 0 or gt_bin.sum() == 0:
        iou = 0.0
    else:
        iou = IoU(gt_bin, ht_bin)

    return {"auc": auc, "f1": f1, "iou": iou}


def evaluate_run(
    heatmaps_path: Path,
    gtmaps_path:   Path,
    labels:        np.ndarray,
    chunk_size:    int = 50,
) -> dict:
    anom_idx    = np.where(labels == 1)[0]
    n_anom      = len(anom_idx)
    print(f"Immagini anomale nel test set: {n_anom}")

    heatmaps_mm = np.load(heatmaps_path, mmap_mode='r')
    gtmaps_mm   = np.load(gtmaps_path,   mmap_mode='r')

    aucs, f1s, ious, sigmas_list = [], [], [], []
    blurred_list, binary_list    = [], []

    for start in range(0, n_anom, chunk_size):
        end      = min(start + chunk_size, n_anom)
        idxs     = anom_idx[start:end]
        ht_chunk = heatmaps_mm[idxs].copy()
        gt_chunk = gtmaps_mm[idxs].copy()

        for i in range(len(idxs)):
            ht2d   = ht_chunk[i, 0]
            gt2d   = gt_chunk[i, 0]
            sigma  = get_filter_sigma(ht2d)
            ht_b   = blur_one(ht2d)
            ht_bin = binarize_heatmap(ht_b)
            m      = compute_metrics_single(ht_b, gt2d)

            aucs.append(m["auc"])
            f1s.append(m["f1"])
            ious.append(m["iou"])
            sigmas_list.append(sigma)
            blurred_list.append(ht_b[np.newaxis])
            binary_list.append(ht_bin[np.newaxis])

        print(f"  chunk {start}-{end} completato")

    aucs    = np.array(aucs)
    f1s     = np.array(f1s)
    ious    = np.array(ious)
    sigmas  = np.array(sigmas_list)
    blurred = np.stack(blurred_list)[:, np.newaxis]
    binary  = np.stack(binary_list)[:, np.newaxis]

    valid = ~np.isnan(aucs)
    if not valid.all():
        print(f"  WARN: {(~valid).sum()} immagini con AUC=nan escluse")

    return {
        "blurred":  blurred,
        "binary":   binary,
        "sigmas":   sigmas,
        "auc":      aucs,
        "f1":       f1s,
        "iou":      ious,
        "mean_auc": float(np.nanmean(aucs)),
        "mean_f1":  float(np.nanmean(f1s)),
        "mean_iou": float(np.nanmean(ious)),
        "std_auc":  float(np.nanstd(aucs)),
        "std_f1":   float(np.nanstd(f1s)),
        "std_iou":  float(np.nanstd(ious)),
    }


# -----------------------------------------------------------------------
# ESECUZIONE SU TUTTI I SEED
# -----------------------------------------------------------------------

for SEED in SEEDS:
    OUT_RUN = RUNS_BASE / f"seed{SEED}"
    print(f"\n{'='*55}")
    print(f"Post-processing e metriche — seed {SEED}")
    print(f"{'='*55}\n")

    labels   = np.load(OUT_RUN / "labels.npy")
    anom_idx = np.where(labels == 1)[0]

    results_seed = evaluate_run(
        heatmaps_path = OUT_RUN / "heatmaps.npy",
        gtmaps_path   = OUT_RUN / "gtmaps.npy",
        labels        = labels,
        chunk_size    = 50,
    )

    print(f"\nRisultati seed {SEED}:")
    print(f"  pixel-AUC:  {results_seed['mean_auc']:.4f} ± {results_seed['std_auc']:.4f}")
    print(f"  F1:         {results_seed['mean_f1']:.4f} ± {results_seed['std_f1']:.4f}")
    print(f"  IoU:        {results_seed['mean_iou']:.4f} ± {results_seed['std_iou']:.4f}")

    np.save(OUT_RUN / "heatmaps_blurred.npy", results_seed["blurred"])
    np.save(OUT_RUN / "heatmaps_binary.npy",  results_seed["binary"])
    print(f"✓ heatmaps_blurred.npy e heatmaps_binary.npy salvati")

    # --- Metriche per classe (benign vs malignant) ---
    with open(OUT_RUN / "test_anom_files.json") as f:
        anom_meta = json.load(f)

    assert len(anom_meta) == len(anom_idx), (
        f"DISALLINEAMENTO test_anom_files.json: {len(anom_meta)} entry "
        f"vs {len(anom_idx)} anomalie in labels."
    )
    assert len(results_seed["auc"]) == len(anom_idx), (
        f"results_seed disallineato: {len(results_seed['auc'])} metriche "
        f"vs {len(anom_idx)} anomalie."
    )

    cls_arr = np.array([e["cls"] for e in anom_meta])
    auc     = results_seed["auc"]
    f1      = results_seed["f1"]
    iou     = results_seed["iou"]

    print(f"\nMetriche per classe — seed {SEED}")
    print(f"{'TOTALE':10s}: n={len(auc):4d}  "
          f"AUC={np.nanmean(auc):.4f} ± {np.nanstd(auc):.4f}  "
          f"F1={np.nanmean(f1):.4f} ± {np.nanstd(f1):.4f}  "
          f"IoU={np.nanmean(iou):.4f} ± {np.nanstd(iou):.4f}")

    per_classe = {}
    for cls in ("benign", "malignant"):
        mask = (cls_arr == cls)
        n    = int(mask.sum())
        if n == 0:
            print(f"{cls:10s}: n=0  (assente in questo split)")
            continue
        rec = {
            "n":       n,
            "auc":     float(np.nanmean(auc[mask])),
            "auc_std": float(np.nanstd(auc[mask])),
            "f1":      float(np.nanmean(f1[mask])),
            "f1_std":  float(np.nanstd(f1[mask])),
            "iou":     float(np.nanmean(iou[mask])),
            "iou_std": float(np.nanstd(iou[mask])),
        }
        per_classe[cls] = rec
        print(f"{cls:10s}: n={n:4d}  "
              f"AUC={rec['auc']:.4f} ± {rec['auc_std']:.4f}  "
              f"F1={rec['f1']:.4f} ± {rec['f1_std']:.4f}  "
              f"IoU={rec['iou']:.4f} ± {rec['iou_std']:.4f}")

    with open(OUT_RUN / "metrics_per_class.json", "w") as fp:
        json.dump({
            "seed":      int(SEED),
            "total":     {"n": len(auc),
                          "auc": float(np.nanmean(auc)),
                          "f1":  float(np.nanmean(f1)),
                          "iou": float(np.nanmean(iou))},
            "per_class": per_classe,
        }, fp, indent=2)
    print(f"✓ metrics_per_class.json salvato in {OUT_RUN}")

In [ ]:
# Aggregazione risultati su tutti i seed completati ===

import json
import csv
import gc
import numpy as np
from pathlib import Path

SEEDS_ALL = [101, 237, 359, 461, 491]
RUNS_DIR  = RUNS_BASE

per_seed_results = {}

# NOTA: l'aggregazione è su medie per seed (non per immagine) e su tutte le anomalie
# aggregate. Per un'analisi separata per classe (benign vs malignant), caricare
# test_anom_files.json da run_dir per ogni seed e filtrare anom_idx per cls.

for seed in SEEDS_ALL:
    run_dir = RUNS_DIR / f"seed{seed}"

    required = [
        run_dir / "heatmaps.npy",
        run_dir / "gtmaps.npy",
        run_dir / "labels.npy",
        run_dir / "config.json",
    ]
    missing = [f.name for f in required if not f.exists()]

    if missing:
        print(f"  seed {seed}: SKIP — file mancanti: {missing}")
        continue

    print(f"  seed {seed}: calcolo metriche...")

    labels = np.load(run_dir / "labels.npy")
    res    = evaluate_run(
        heatmaps_path = run_dir / "heatmaps.npy",
        gtmaps_path   = run_dir / "gtmaps.npy",
        labels        = labels,
        chunk_size    = 50,
    )

    # Salviamo solo i numeri: medie, std e vettori per-immagine (pochi KB).
    # blurred e binary pesano ~1.3 GB per seed: tenerli tutti in memoria
    # contemporaneamente causerebbe OOM. Verranno ricalcolati in cella 23
    # solo per il seed scelto per le visualizzazioni.
    per_seed_results[seed] = {
        "mean_auc": res["mean_auc"],
        "mean_f1":  res["mean_f1"],
        "mean_iou": res["mean_iou"],
        "std_auc":  res["std_auc"],
        "std_f1":   res["std_f1"],
        "std_iou":  res["std_iou"],
        "auc":      res["auc"],
        "f1":       res["f1"],
        "iou":      res["iou"],
    }

    # Libera blurred e binary prima del seed successivo
    del res
    gc.collect()

    r = per_seed_results[seed]
    print(f"    AUC {r['mean_auc']:.4f}  F1 {r['mean_f1']:.4f}  IoU {r['mean_iou']:.4f}")

# -----------------------------------------------------------------------

n_completed = len(per_seed_results)
print(f"\nSeed completati: {n_completed} / {len(SEEDS_ALL)}")

if n_completed == 0:
    print("Nessun seed completato. Esegui prima la cella 19 per almeno un seed.")
else:
    # Aggregazione su medie per seed (non per immagine — vedi cella 21)
    mean_aucs = np.array([r["mean_auc"] for r in per_seed_results.values()])
    mean_f1s  = np.array([r["mean_f1"]  for r in per_seed_results.values()])
    mean_ious = np.array([r["mean_iou"] for r in per_seed_results.values()])

    agg = {
        "n_seeds_completed": n_completed,
        "seeds": list(per_seed_results.keys()),
        "per_seed": {
            str(seed): {
                "auc": round(r["mean_auc"], 4),
                "f1":  round(r["mean_f1"],  4),
                "iou": round(r["mean_iou"], 4),
            }
            for seed, r in per_seed_results.items()
        },
        "aggregate": {
            "mean_auc": round(float(mean_aucs.mean()), 4),
            "std_auc":  round(float(mean_aucs.std()),  4),
            "mean_f1":  round(float(mean_f1s.mean()),  4),
            "std_f1":   round(float(mean_f1s.std()),   4),
            "mean_iou": round(float(mean_ious.mean()), 4),
            "std_iou":  round(float(mean_ious.std()),  4),
        }
    }

    # Salvataggio JSON
    results_json = RUNS_DIR / "results.json"
    results_json.write_text(json.dumps(agg, indent=2))

    # Salvataggio CSV: una riga per seed + riga aggregata finale
    results_csv = RUNS_DIR / "results.csv"
    with open(results_csv, "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(["seed", "mean_auc", "mean_f1", "mean_iou"])
        for seed, r in per_seed_results.items():
            writer.writerow([seed,
                             round(r["mean_auc"], 4),
                             round(r["mean_f1"],  4),
                             round(r["mean_iou"], 4)])
        writer.writerow([
            "mean ± std",
            f"{mean_aucs.mean():.4f} ± {mean_aucs.std():.4f}",
            f"{mean_f1s.mean():.4f} ± {mean_f1s.std():.4f}",
            f"{mean_ious.mean():.4f} ± {mean_ious.std():.4f}",
        ])

    # Stampa riepilogo
    print("\n" + "="*50)
    print("RISULTATI AGGREGATI")
    print("="*50)
    print(f"  {'Seed':<8} {'AUC':>8} {'F1':>8} {'IoU':>8}")
    print(f"  {'-'*36}")
    for seed, r in per_seed_results.items():
        print(f"  {str(seed):<8} {r['mean_auc']:>8.4f} {r['mean_f1']:>8.4f} {r['mean_iou']:>8.4f}")
    print(f"  {'-'*36}")
    a = agg["aggregate"]
    print(f"  {'μ±σ':<8} "
          f"{a['mean_auc']:.4f}±{a['std_auc']:.4f}  "
          f"{a['mean_f1']:.4f}±{a['std_f1']:.4f}  "
          f"{a['mean_iou']:.4f}±{a['std_iou']:.4f}")

    print(f"\n✓ results.json e results.csv salvati in {RUNS_DIR}")

    if n_completed < len(SEEDS_ALL):
        remaining = [s for s in SEEDS_ALL if s not in per_seed_results]
        print(f"\nSeed rimanenti: {remaining}")
        print("Per ognuno: modifica SEED in cella 18 e riesegui celle 18 e 19.")

In [ ]:
# Visualizzazioni qualitative ===

import json
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from pathlib import Path
from PIL import Image as PILImage
from scipy.ndimage import gaussian_filter
from torchvision import transforms
import torch

# --- Seed con AUC più alta ---
# Mostriamo il caso migliore: se anche il best seed produce heatmaps
# mediocri, è un'informazione rilevante. Se usassimo il seed peggiore,
# il commissario potrebbe obiettare che abbiamo scelto il caso sfortunato.
best_seed = max(per_seed_results, key=lambda s: per_seed_results[s]["mean_auc"]) #seed migliore
#best_seed = SEED   # mostra il seed corrente 
run_dir   = RUNS_DIR / f"seed{best_seed}"
print(f"Seed scelto: {best_seed} (AUC {per_seed_results[best_seed]['mean_auc']:.4f})")

aucs     = per_seed_results[best_seed]["auc"]
labels   = np.load(run_dir / "labels.npy")
anom_idx = np.where(labels == 1)[0]


# --- Selezione 6 esempi: 2 best, 2 median, 2 worst ---
# Escludiamo nan (immagini con GT costante, roc_auc_score non definita).
# Mostrare best/median/worst dà al commissario una visione onesta
# della distribuzione delle prestazioni, non solo i casi favorevoli.
valid_mask = ~np.isnan(aucs)
valid_pos  = np.where(valid_mask)[0]
sorted_pos = valid_pos[np.argsort(aucs[valid_mask])]
n_valid    = len(sorted_pos)
mid        = n_valid // 2

selected_anom_pos = np.array([
    sorted_pos[-1], sorted_pos[-2],
    sorted_pos[mid], sorted_pos[mid - 1],
    sorted_pos[1],  sorted_pos[0],
])
selected_test_idx = anom_idx[selected_anom_pos]

labels_str = [
    f"Best\nAUC={aucs[selected_anom_pos[0]]:.3f}",
    f"2nd best\nAUC={aucs[selected_anom_pos[1]]:.3f}",
    f"Median\nAUC={aucs[selected_anom_pos[2]]:.3f}",
    f"Median-1\nAUC={aucs[selected_anom_pos[3]]:.3f}",
    f"2nd worst\nAUC={aucs[selected_anom_pos[4]]:.3f}",
    f"Worst\nAUC={aucs[selected_anom_pos[5]]:.3f}",
]

# --- Mapping indice → file originale ---
# Carichiamo test_anom_files.json da run_dir su Drive, non da NPY_DIR:
# dopo un reset della sessione NPY_DIR (/content/) non esiste più,
# mentre Drive è sempre accessibile.
# Il json è seed-specifico: seed diversi → shuffle diverso → ordine diverso.
# Usare split_info.json (ordine alfabetico) sarebbe sbagliato perché
# extract_dataset_gray mescola i file con rng.permutation(seed),
# producendo un ordine diverso da quello alfabetico.
test_anom_files = json.loads(
    (run_dir / "test_anom_files.json").read_text()
)

# --- Caricamento modello per le ricostruzioni ---
# map_location="cpu": carica il checkpoint sulla CPU indipendentemente
# da dove è stato salvato. Senza questo argomento, su una sessione
# senza GPU attiva crasherebbe con RuntimeError: CUDA not available.
print("Caricamento modello...")
model = ResNet_CNN_Attn_Gray(dim=[1])
model.load_state_dict(torch.load(
    run_dir / "aexad_model_vgg_cnn_gray.pt",
    map_location="cpu"
))
model.eval()
if torch.cuda.is_available():
    model = model.cuda()
print("✓ Modello caricato")

# Pipeline identica al dataloader: stesso resize e normalizzazione.
# Se usassimo una pipeline diversa, la ricostruzione non sarebbe
# confrontabile con l'input visto durante il training.
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.0], [1.0]),
])

# --- Raccolta dati per i 6 esempi (un'immagine alla volta) ---
# mmap_mode='r': il file resta su disco, in RAM va solo la slice richiesta.
# Con 6 immagini su 681, caricare tutto sarebbe uno spreco di ~700 MB.
heatmaps_mm = np.load(run_dir / "heatmaps.npy", mmap_mode='r')
gtmaps_mm   = np.load(run_dir / "gtmaps.npy",   mmap_mode='r')

examples = []
for anom_pos, test_idx in zip(selected_anom_pos, selected_test_idx):

    # File corretto per questo indice, garantito dal json seed-specifico.
    # entry["cls"] e entry["fname"] sono stati scritti nell'ordine esatto
    # di inserimento in X_test da extract_dataset_gray.
    entry    = test_anom_files[anom_pos]
    cls      = entry["cls"]
    fname    = entry["fname"]
    img_path = OUT_DIR / "test" / cls / "images" / fname

    # Input originale: uint8 → float [0,1] per imshow
    pil_img = PILImage.open(img_path).convert("L")
    img_arr = np.array(pil_img, dtype=np.float32) / 255.0

    # Ricostruzione via forward pass.
    # Con mean=0.0, std=1.0 l'input è in [0,1] e l'output pure.
    # Nessuna denormalizzazione necessaria.

    tensor_in = transform(pil_img).unsqueeze(0)
    if torch.cuda.is_available():
        tensor_in = tensor_in.cuda()
    with torch.no_grad():
        tensor_out = model(tensor_in)
    recon = tensor_out[0, 0].cpu().numpy()
    recon = np.clip(recon, 0, 1)

    # Heatmap: ricalcoliamo filtro e binarizzazione puntualmente.
    # Non carichiamo heatmaps_blurred.npy (641 immagini) per 6 esempi:
    # il ricalcolo è istantaneo e risparmia ~650 MB di RAM.
    ht_raw  = heatmaps_mm[test_idx, 0].copy()
    gt_raw  = gtmaps_mm[test_idx, 0].copy()
    sigma   = get_filter_sigma(ht_raw)
    ht_blur = gaussian_filter(ht_raw, sigma=sigma)
    ht_bin  = binarize_heatmap(ht_blur)
    gt_bin  = (gt_raw > 0).astype(np.float32)

    examples.append({
        "img":     img_arr,
        "recon":   recon,
        "ht_raw":  ht_raw,
        "ht_blur": ht_blur,
        "ht_bin":  ht_bin.astype(np.float32),
        "gt":      gt_bin,
    })

# Liberiamo memoria prima di costruire la figura.
# Il modello da solo occupa ~200 MB in VRAM.
del heatmaps_mm, gtmaps_mm, model
print("✓ Dati raccolti per tutti gli esempi")

# --- Figura 6 righe × 6 colonne ---
col_titles = ["Input", "Ricostruzione", "Heatmap grezza", "Filtrata", "Binarizzata", "GT"]
n_cols     = len(col_titles)
n_rows     = len(examples)

fig = plt.figure(figsize=(n_cols * 2.8, n_rows * 2.8))
gs  = gridspec.GridSpec(n_rows, n_cols, figure=fig, hspace=0.35, wspace=0.05)

for j, title in enumerate(col_titles):
    ax = fig.add_subplot(gs[0, j])
    ax.set_title(title, fontsize=10, fontweight='bold', pad=6)

for i, (ex, lbl) in enumerate(zip(examples, labels_str)):
    panels = [
        (ex["img"],     'gray',   0, 1),
        (ex["recon"],   'gray',   0, 1),
        (ex["ht_raw"],  'jet',    None, None),
        (ex["ht_blur"], 'jet',    None, None),
        (ex["ht_bin"],  'gray', 0, 1),
        (ex["gt"],      'gray', 0, 1),
    ]
    for j, (data, cmap, vmin, vmax) in enumerate(panels):
        ax = fig.add_subplot(gs[i, j])
        ax.set_xticks([])
        ax.set_yticks([])
        if j == 0:
            ax.set_ylabel(lbl, fontsize=8, rotation=0,
                          labelpad=90, va='center')
        ax.imshow(data, cmap=cmap, vmin=vmin, vmax=vmax)

fig.suptitle(
    f"Esempi qualitativi — seed {best_seed}  "
    f"(AUC medio {per_seed_results[best_seed]['mean_auc']:.4f})",
    fontsize=12, fontweight='bold', y=1.01
)

out_path = RUNS_DIR / "qualitative.png"
fig.savefig(out_path, dpi=150, bbox_inches='tight')
plt.show()
print(f"\n✓ qualitative.png salvato in {out_path}")

In [ ]:
FIX GT: sostituisce nei gtmaps.npy le maschere parziali con quelle fuse ===
# Lancia DOPO le celle di setup (0-6). Solo per i seed NON riaddestrati.
import json, numpy as np
from pathlib import Path
from PIL import Image

SEEDS_DA_FIXARE = [101, 359, 491]      # i 3 buoni; 237/461 vengono riaddestrati
RAW = BUSI_RAW / "Dataset_BUSI_with_GT"  # cartella con benign/ malignant/ raw

def mask_fusa_224(cls: str, fname: str) -> np.ndarray:
    """Ricarica la maschera RAW fusa (load_mask già fa l'OR) e la porta a 224x224 NEAREST,
    identico a save_mask del preprocessing. Ritorna (1,224,224) uint8 con {0,255}."""
    img_path = RAW / cls / fname
    m = load_mask(img_path)                       # OR delle _mask*, valori {0,1}, shape HxW
    if m is None:
        raise ValueError(f"maschera vuota/assente per {cls}/{fname}")
    m255 = (m * 255).astype(np.uint8)
    m_res = np.array(Image.fromarray(m255).resize(RESIZE_TO, Image.NEAREST), dtype=np.uint8)
    return m_res[np.newaxis, :, :]                 # (1,224,224)

for SEED in SEEDS_DA_FIXARE:
    OUT_RUN = RUNS_BASE / f"seed{SEED}"
    gt_path = OUT_RUN / "gtmaps.npy"
    taf     = json.loads((OUT_RUN / "test_anom_files.json").read_text())

    gt = np.load(gt_path)                          # (N,1,224,224)
    labels = np.load(OUT_RUN / "labels.npy")
    anom_idx = np.where(labels == 1)[0]            # indici delle anomale in gt

    # allineamento: la i-esima entry di taf corrisponde alla i-esima anomala
    assert len(taf) == len(anom_idx), \
        f"seed {SEED}: DISALLINEAMENTO taf({len(taf)}) vs anomale({len(anom_idx)})"

    cambiati = 0
    for i, entry in enumerate(taf):
        cls, fname = entry["cls"], entry["fname"]
        # ha una maschera multipla?  (esiste un _mask_1 ecc.)
        stem = Path(fname).stem
        raw_folder = RAW / cls
        has_multi = any(raw_folder.glob(f"{stem}_mask_*.png"))
        if not has_multi:
            continue
        gidx = anom_idx[i]                          # indice reale dentro gt
        old_px = int((gt[gidx, 0] > 0).sum())
        gt[gidx] = mask_fusa_224(cls, fname)
        new_px = int((gt[gidx, 0] > 0).sum())
        print(f"  seed {SEED}: {cls}/{fname}  {old_px} -> {new_px} px")
        cambiati += 1

    # controllo di sicurezza: nessuna riga non-target deve essere cambiata di shape/conteggio
    assert gt.shape[0] == len(labels), f"seed {SEED}: shape gt alterata!"
    np.save(gt_path, gt)
    print(f"seed {SEED}: {cambiati} maschere fuse sovrascritte in gtmaps.npy\n")

print("FATTO. Ora rilancia la CELLA 24 (post-processing) per ricalcolare le metriche.")